# Real-wheel zero-shot pilot — data, crops and frozen models

This notebook prepares the held-out MAHLI pilot and carries the same four model
families used in the synthetic roadmap: PatchCore, EfficientAD-S,
SuperSimpleNet and TinyGLASS. Change only `MODEL_NAME` to select the family;
the corresponding best verified preset, input geometry and normalization are
derived automatically.

Experimental contract:

- Kaggle mounts the fixed dataset directly at
  `/kaggle/input/datasets/dalphan01/real-mars-rover-wheel-ad/real_wheel_pilot_v2`; no root discovery is performed;
- the 20 real images are evaluation-only; do not train, fit, calibrate, select
  checkpoints or tune thresholds on them;
- checkpoint selection is frozen from the synthetic experiments and verified
  against the Drive manifest before any real-image prediction;
- use a final `model.ckpt`; training checkpoints such as `*_training.ckpt` are
  deliberately rejected;
- crop coordinates come from visible wheel geometry, not from hole masks or
  anomaly scores;
- RGB and masks receive identical crop/resize geometry; RGB uses bilinear
  antialiasing and masks use nearest-neighbour interpolation;
- the masks are approximate localization aids, not authoritative segmentation
  ground truth;
- images from the same Sol are correlated, so retain per-image scores and
  heatmaps instead of treating the 20 frames as independent observations.


In [ ]:
# Kaggle setup: install Drive dependencies only when the runtime lacks them.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("googleapiclient") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "google-api-python-client",
        "google-auth",
    ])


In [ ]:
# Configuration — MODEL_NAME is the single model-family selector
from pathlib import Path

MODEL_NAME = "tinyglass"  # patchcore | efficientad_s | supersimplenet | tinyglass
CHECKPOINT_SOURCE = "google_drive"  # google_drive | local | none
MODEL_CHECKPOINT_PATH = None  # used only when CHECKPOINT_SOURCE == "local"
DRIVE_RUN_FOLDER_ID_OVERRIDE = None  # optional raw folder ID or Drive folder URL
DRIVE_CHECKPOINT_CACHE_DIR = Path("/kaggle/working/real_wheel_checkpoints")
SUPPORTED_MODELS = ("patchcore", "efficientad_s", "supersimplenet", "tinyglass")

DATASET_ROOT = Path("/kaggle/input/datasets/dalphan01/real-mars-rover-wheel-ad/real_wheel_pilot_v2")
BATCH_SIZE = 1

NUM_WORKERS = 2
RANDOM_SEED = 42

OUTPUT_ROOT = Path("/kaggle/working/real_wheel_zero_shot")
METRIC_HISTOGRAM_BINS = 2048
PRO_BINS = 256
PRO_MAX_FPR = 0.30
VISUALIZATION_ROWS_PER_PAGE = 4
VISUALIZATION_COLORMAP = "magma"
VISUALIZATION_OVERLAY_ALPHA = 0.55
VISUALIZATION_DPI = 150
FORCE_CHECKPOINT_DOWNLOAD = False

# PatchCore light remains available for a local checkpoint. No final PatchCore
# model.ckpt was found in the accessible Drive anomaly_detection hierarchy.
PATCHCORE_BACKBONE = "resnet18"
PATCHCORE_PRETRAINED = True
PATCHCORE_LAYERS = ("layer2", "layer3")
PATCHCORE_CORESET_SAMPLING_RATIO = 0.1
PATCHCORE_NUM_NEIGHBORS = 9
PATCHCORE_PATCH_SIZE = 3
PATCHCORE_PATCH_STRIDE = 1
PATCHCORE_PRETRAIN_EMBED_DIMENSION = 384
PATCHCORE_TARGET_EMBED_DIMENSION = 384
PATCHCORE_MAX_PATCHES_PER_IMAGE = 128
PATCHCORE_MAX_TRAINING_EMBEDDINGS = 50_000
PATCHCORE_MAX_MEMORY_BANK_SIZE = 2_048
PATCHCORE_PROJECTION_DIM = 64
PATCHCORE_SAMPLING_SEED = RANDOM_SEED
PATCHCORE_CALIBRATION_QUANTILE = 0.99
PATCHCORE_CALIBRATION_BATCHES = 20
PATCHCORE_DISTANCE_QUERY_CHUNK_SIZE = 512
PATCHCORE_DISTANCE_BANK_CHUNK_SIZE = 2_048
PATCHCORE_GAUSSIAN_SIGMA = 4.0

# Configurations frozen from the actual manifests of the recommended Drive runs.
# `batch_size` records the training setting; this pilot always uses batch 1 and
# never calls fit().
MODEL_CONFIGS = {
    "patchcore": {
        "input_size": (384, 512),
        "batch_size": 4,
    },
    "efficientad_s": {
        "input_size": (256, 256),
        "teacher_weights_path": None,
        "require_teacher_weights": True,
        "channels": 384,
        "max_steps": 70_000,
        "learning_rate": 1e-4,
        "weight_decay": 1e-5,
        "hard_quantile": 0.995,
        "checkpoint_interval": 1_000,
        "validation_interval": 1_000,
        "early_stopping_patience": 8,
        "early_stopping_min_steps": 10_000,
        "early_stopping_min_relative_improvement": 0.001,
        "lr_scheduler_patience": 3,
        "fixed_training_duration": True,
        "mixed_precision": True,
        "spatial_calibration_enabled": False,
        "spatial_calibration_q_low": 0.90,
        "spatial_calibration_q_high": 0.995,
        "spatial_calibration_smoothing_sigma": 1.0,
        "spatial_scale_floor_fraction": 0.25,
        "static_roi_min_coverage": 0.50,
        "image_score_topk_candidates": (0.0,),
        "batch_size": 1,
    },
    "supersimplenet": {
        "backbone": "wide_resnet50_2",
        "pretrained": True,
        "weights_name": "IMAGENET1K_V1",
        "layers": ("layer2", "layer3"),
        "input_size": (256, 256),
        "patch_size": 3,
        "epochs": 100,
        "noise_std": 0.015,
        "perlin_threshold": 0.2,
        "restrict_synthetic_anomalies_to_target_mask": False,
        "adaptor_learning_rate": 1e-4,
        "segmentation_learning_rate": 2e-4,
        "decision_learning_rate": 2e-4,
        "scheduler_gamma": 0.4,
        "stop_grad": True,
        "adapt_classification_features": False,
        "gradient_clip": False,
        "margin": 0.5,
        "gaussian_sigma": 4.0,
        "image_score_mode": "topk_fraction_mean",
        "image_score_fraction": 0.01,
        "fixed_training_duration": False,
        "validation_interval": 4,
        "validation_batches": None,
        "early_stopping_patience": 5,
        "early_stopping_min_delta": 0.001,
        "max_samples_per_epoch": 1_024,
        "checkpoint_interval": 1,
        "batch_size": 16,
    },
    "tinyglass": {
        "pretrained": True,
        "weights_name": "IMAGENET1K_V1",
        "input_size": (384, 384),
        "patch_size": 3,
        "epochs": 640,
        "learning_rate": 5e-5,
        "weight_decay": 1e-2,
        "noise_std": 0.015,
        "radius_quantile": 0.75,
        "hard_mining_quantile": 0.5,
        "gas_steps": 20,
        "gas_step_size": 0.001,
        "hypersphere_projection": False,
        "feature_grid_resolution": "layer2",
        "max_samples_per_epoch": 392,
        "texture_root": None,
        "require_texture_dataset": False,
        "blend_mean": 0.5,
        "blend_std": 0.1,
        "gaussian_sigma": 1.0,
        "las_restrict_to_target_mask": True,
        "las_mode": "hole",
        "las_hole_probability": 0.5,
        "las_hole_luminance_range": (0.18, 0.32),
        "las_hole_severity_weights": (0.70, 0.25, 0.05),
        "freeze_backbone": True,
        "backbone_learning_rate": 1e-5,
        "fixed_training_duration": False,
        "validation_interval": 1,
        "validation_batches": None,
        "cache_validation_features": True,
        "early_stopping_patience": 40,
        "early_stopping_min_delta": 0.0,
        "checkpoint_interval": 1,
        "batch_size": 8,
    },
}

# Verified run folders. Folder names were not trusted: each entry was checked
# against run_manifest.json, metrics.json and the presence of one model.ckpt.
DRIVE_RUN_FOLDER_CATALOG = {
    "efficientad_global_256_recommended": {
        "model": "efficientad_s",
        "folder_id": "1LEskJG5sIdUBHIfws1keaaZxArvD-WWU",
        "checkpoint_file_id": "1LTxIQmEovzJTnosP6Plwrobvh7qBpmr5",
        "input_size": (256, 256),
        "validation_image_auroc": 0.8123240428,
        "validation_target_wheel_pixel_ap": 0.1645902868,
        "compatible_with_current_preset": True,
        "comment": "Recommended: global fixed-70k; strongest EfficientAD localization/generalization.",
    },
    "efficientad_spatial_256_alternative": {
        "model": "efficientad_s",
        "folder_id": "1-M2ikLWC9OnTJOm1PjBVJxgs7oVcfCk4",
        "checkpoint_file_id": "18Qpn6t9fTeRbeaTgdWwMBQf801LPvgG1",
        "input_size": (256, 256),
        "validation_image_auroc": 0.8500140766,
        "validation_target_wheel_pixel_ap": 0.0605935599,
        "compatible_with_current_preset": False,
        "comment": "Alternative spatial calibration: higher validation Image-AUROC, much weaker localization.",
    },
    "supersimplenet_perlin02_unrestricted_recommended": {
        "model": "supersimplenet",
        "folder_id": "1XPHgLw2jVMiRJRtwouqtIWFrQeEHDBeA",
        "checkpoint_file_id": "1uAdzWRKpmQZnjECnAuEXD08CtDY8xfhv",
        "input_size": (256, 256),
        "validation_image_auroc": 0.9334881757,
        "validation_target_wheel_pixel_ap": 0.1848988693,
        "compatible_with_current_preset": True,
        "comment": "Recommended compromise: Perlin 0.2, top-1% score, unrestricted synthetic anomalies.",
    },
    "tinyglass_384_layer2_recommended": {
        "model": "tinyglass",
        "folder_id": "1Xo5ELdDZdMKsfhKjuDT2HaQm4RqyTonb",
        "checkpoint_file_id": "1L45AfJ4ZeFceXfy2houemOVc_3Kv4Qpr",
        "input_size": (384, 384),
        "validation_image_auroc": 0.8835515203,
        "validation_target_wheel_pixel_ap": 0.2958957823,
        "compatible_with_current_preset": True,
        "comment": "Recommended: best TinyGLASS validation Image-AUROC and wheel pixel AP among the resolution/grid runs.",
    },
    "tinyglass_384_layer3_alternative": {
        "model": "tinyglass",
        "folder_id": "1ft0zDYSQrwE9eomJ4Ksg1LmLmWr_NZt4",
        "checkpoint_file_id": "1CfadL9Uo32gwWOjZCqt6WcYh0nWneeVo",
        "input_size": (384, 384),
        "validation_image_auroc": 0.8757742117,
        "validation_target_wheel_pixel_ap": 0.2704437951,
        "compatible_with_current_preset": False,
        "comment": "Alternative 384/layer3; requires feature_grid_resolution='layer3'.",
    },
    "tinyglass_512_layer2_alternative": {
        "model": "tinyglass",
        "folder_id": "1aAs2KVGiwNZ0zfHEk_8nHSGOyp9gAxOW",
        "checkpoint_file_id": "1d2TwdylOhW6bCZEpxNOEywZt4fSh1Os0",
        "input_size": (512, 512),
        "validation_image_auroc": 0.8316265484,
        "validation_target_wheel_pixel_ap": 0.2531072812,
        "compatible_with_current_preset": False,
        "comment": "Alternative 512/layer2; requires input_size=(512, 512).",
    },
    "tinyglass_512_layer3_alternative": {
        "model": "tinyglass",
        "folder_id": "1hWFs0qKm4xqGiT98ZH8XAJCDTl06u5iL",
        "checkpoint_file_id": "1gq7zyfAJZ5Vza2GjLBsIA91lVKU-b3LW",
        "input_size": (512, 512),
        "validation_image_auroc": 0.8556798986,
        "validation_target_wheel_pixel_ap": 0.2907638596,
        "compatible_with_current_preset": False,
        "comment": "Alternative 512/layer3; folder name is stale and incorrectly says 384/layer2.",
    },
}

BEST_DRIVE_RUN_KEYS = {
    "patchcore": None,
    "efficientad_s": "efficientad_global_256_recommended",
    "supersimplenet": "supersimplenet_perlin02_unrestricted_recommended",
    "tinyglass": "tinyglass_384_layer2_recommended",
}

if MODEL_NAME not in SUPPORTED_MODELS:
    raise ValueError(f"MODEL_NAME must be one of {SUPPORTED_MODELS}, got {MODEL_NAME!r}")
if CHECKPOINT_SOURCE not in {"google_drive", "local", "none"}:
    raise ValueError("CHECKPOINT_SOURCE must be google_drive, local, or none")
SELECTED_MODEL_CONFIG = MODEL_CONFIGS[MODEL_NAME]
MODEL_INPUT_SIZE = tuple(SELECTED_MODEL_CONFIG["input_size"])
NORMALIZE_IMAGENET = MODEL_NAME != "efficientad_s"
SELECTED_DRIVE_RUN_KEY = BEST_DRIVE_RUN_KEYS[MODEL_NAME]
SELECTED_DRIVE_RUN = (
    None
    if SELECTED_DRIVE_RUN_KEY is None
    else DRIVE_RUN_FOLDER_CATALOG[SELECTED_DRIVE_RUN_KEY]
)


## Verified Google Drive checkpoint folders

The catalog in `Configuration` contains the raw folder IDs, exact checkpoint
file IDs and synthetic validation metrics. The download separately checks the
remote byte size. Recommended defaults are selected before seeing the real pilot:

- EfficientAD-S: `1LEskJG5sIdUBHIfws1keaaZxArvD-WWU` — global 256, fixed
  70k, recommended for localization and overall generalization;
- SuperSimpleNet: `1XPHgLw2jVMiRJRtwouqtIWFrQeEHDBeA` — input 256,
  Perlin 0.2, top-1% image score, unrestricted synthetic anomalies;
- TinyGLASS: `1Xo5ELdDZdMKsfhKjuDT2HaQm4RqyTonb` — input 384,
  layer2, sigma 1; best validation Image-AUROC and wheel pixel AP of the
  resolution/grid comparison;
- PatchCore: no final Drive checkpoint was found; use `CHECKPOINT_SOURCE="local"`
  with a matching `MODEL_CHECKPOINT_PATH`.

Verified alternatives remain in `DRIVE_RUN_FOLDER_CATALOG`, but are deliberately
marked incompatible with the current preset when they require different input,
feature grid or calibration. A manual folder override never disables strict
configuration checking.


In [ ]:
# Imports and deterministic setup
from __future__ import annotations

import copy
import csv
import json
import math
import os
import random
import warnings
from collections import deque
from collections.abc import Callable, Iterable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
from matplotlib.patches import Rectangle
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import get_model, get_model_weights
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from torchvision.transforms.functional import gaussian_blur, pil_to_tensor
from tqdm.auto import tqdm

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
RGBTriplet = tuple[float, float, float]
print(
    f"torch={torch.__version__} | device={DEVICE} | model={MODEL_NAME} | "
    f"input={MODEL_INPUT_SIZE} | external_imagenet_norm={NORMALIZE_IMAGENET}"
)


## Four frozen model families

The following two cells are copied from the authoritative model-roadmap
notebook so class definitions, prediction outputs and checkpoint contracts stay
identical. Training methods remain present for checkpoint compatibility, but
this notebook never invokes them on the real pilot.


In [ ]:
@dataclass(frozen=True)
class AnomalyPrediction:
    anomaly_score: torch.Tensor
    anomaly_map: torch.Tensor

    def __post_init__(self):
        if self.anomaly_score.ndim != 1:
            raise ValueError("anomaly_score must have shape [B]")
        if self.anomaly_map.ndim != 4 or self.anomaly_map.shape[1] != 1:
            raise ValueError("anomaly_map must have shape [B, 1, H, W]")
        if self.anomaly_score.shape[0] != self.anomaly_map.shape[0]:
            raise ValueError("anomaly_score and anomaly_map batch sizes must match")
        for name, value in (("anomaly_score", self.anomaly_score), ("anomaly_map", self.anomaly_map)):
            if not value.is_floating_point() or not torch.isfinite(value).all():
                raise ValueError(f"{name} must contain finite floating-point values")
            if value.numel() and (value.min() < 0 or value.max() > 1):
                raise ValueError(f"{name} must be normalized to [0, 1]")


class AnomalyDetector(nn.Module):
    @property
    def is_fitted(self):
        raise NotImplementedError

    def fit(self, train_loader: Iterable[Mapping[str, Any]], *, device):
        raise NotImplementedError

    @torch.no_grad()
    def predict(self, images):
        raise NotImplementedError

    def checkpoint_config(self):
        return {}

    def save(self, path, *, metadata=None):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            "schema_version": 2,
            "model_class": f"{type(self).__module__}.{type(self).__qualname__}",
            "model_config": self.checkpoint_config(),
            "model_state_dict": self.state_dict(),
            "metadata": dict(metadata or {}),
        }, path)
        return path

    def _prepare_state_dict_for_load(self, state_dict):
        pass

    def load(self, path, *, map_location=None):
        payload = torch.load(Path(path), map_location=map_location)
        if not isinstance(payload, dict) or "model_state_dict" not in payload:
            raise ValueError("Checkpoint does not contain model_state_dict")
        if payload.get("schema_version") != 2:
            raise ValueError(f"Unsupported checkpoint schema {payload.get('schema_version')!r}")
        expected_class = f"{type(self).__module__}.{type(self).__qualname__}"
        if payload.get("model_class") != expected_class:
            raise ValueError(
                f"Checkpoint model class {payload.get('model_class')!r} does not "
                f"match {expected_class!r}"
            )
        saved_config = payload.get("model_config")
        expected_config = self.checkpoint_config()
        if saved_config != expected_config:
            raise ValueError(
                "Checkpoint model configuration does not match the current model: "
                f"saved={saved_config!r}, current={expected_config!r}"
            )
        state_dict = payload["model_state_dict"]
        self._prepare_state_dict_for_load(state_dict)
        self.load_state_dict(state_dict)
        return dict(payload.get("metadata", {}))

In [ ]:
from __future__ import annotations

class PatchCore(AnomalyDetector):
    """Memory-bounded PatchCore anomaly detector with a frozen backbone."""

    def __init__(
        self,
        *,
        backbone: str = "resnet18",
        pretrained: bool = True,
        layers: Sequence[str] = ("layer2", "layer3"),
        coreset_sampling_ratio: float = 0.1,
        num_neighbors: int = 9,
        patch_size: int = 3,
        patch_stride: int = 1,
        pretrain_embed_dimension: int = 384,
        target_embed_dimension: int = 384,
        max_patches_per_image: int = 128,
        max_training_embeddings: int = 50_000,
        max_memory_bank_size: int = 2_048,
        projection_dim: int = 64,
        sampling_seed: int = 42,
        calibration_quantile: float = 0.99,
        calibration_batches: int = 20,
        distance_query_chunk_size: int = 512,
        distance_bank_chunk_size: int = 2_048,
        gaussian_sigma: float = 4.0,
    ) -> None:
        super().__init__()
        if not layers:
            raise ValueError("layers must contain at least one feature node")
        if not 0 < coreset_sampling_ratio <= 1:
            raise ValueError("coreset_sampling_ratio must be in (0, 1]")
        if num_neighbors < 1:
            raise ValueError("num_neighbors must be at least 1")
        if patch_size < 1 or patch_size % 2 == 0:
            raise ValueError("patch_size must be a positive odd integer")
        for name, value in (
            ("max_patches_per_image", max_patches_per_image),
            ("max_training_embeddings", max_training_embeddings),
            ("max_memory_bank_size", max_memory_bank_size),
            ("projection_dim", projection_dim),
            ("patch_stride", patch_stride),
            ("pretrain_embed_dimension", pretrain_embed_dimension),
            ("target_embed_dimension", target_embed_dimension),
            ("calibration_batches", calibration_batches),
            ("distance_query_chunk_size", distance_query_chunk_size),
            ("distance_bank_chunk_size", distance_bank_chunk_size),
        ):
            if value < 1:
                raise ValueError(f"{name} must be at least 1")
        if not 0 < calibration_quantile <= 1:
            raise ValueError("calibration_quantile must be in (0, 1]")
        if gaussian_sigma < 0:
            raise ValueError("gaussian_sigma cannot be negative")

        weights = get_model_weights(backbone).DEFAULT if pretrained else None
        backbone_model = get_model(backbone, weights=weights)
        self.feature_extractor = create_feature_extractor(
            backbone_model,
            return_nodes={layer: layer for layer in layers},
        )
        self.feature_extractor.requires_grad_(False)

        self.backbone = backbone
        self.pretrained = bool(pretrained)
        self.layers = tuple(layers)
        self.coreset_sampling_ratio = float(coreset_sampling_ratio)
        self.num_neighbors = int(num_neighbors)
        self.patch_size = int(patch_size)
        self.patch_stride = int(patch_stride)
        self.pretrain_embed_dimension = int(pretrain_embed_dimension)
        self.target_embed_dimension = int(target_embed_dimension)
        self.max_patches_per_image = int(max_patches_per_image)
        self.max_training_embeddings = int(max_training_embeddings)
        self.max_memory_bank_size = int(max_memory_bank_size)
        self.projection_dim = int(projection_dim)
        self.sampling_seed = int(sampling_seed)
        self.calibration_quantile = float(calibration_quantile)
        self.calibration_batches = int(calibration_batches)
        self.distance_query_chunk_size = int(distance_query_chunk_size)
        self.distance_bank_chunk_size = int(distance_bank_chunk_size)
        self.gaussian_sigma = float(gaussian_sigma)

        self.register_buffer("memory_bank", torch.empty(0, 0), persistent=True)
        self.register_buffer("image_score_scale", torch.tensor(0.0), persistent=True)
        self.register_buffer("anomaly_map_scale", torch.tensor(0.0), persistent=True)
        self.fit_summary: dict[str, int | float | list[int]] = {}
        self.train(False)

    @property
    def is_fitted(self) -> bool:
        return (
            self.memory_bank.ndim == 2
            and self.memory_bank.shape[0] > 0
            and self.image_score_scale.item() > 0
            and self.anomaly_map_scale.item() > 0
        )

    def checkpoint_config(self) -> dict[str, object]:
        return {
            "backbone": self.backbone,
            "layers": list(self.layers),
            "coreset_sampling_ratio": self.coreset_sampling_ratio,
            "num_neighbors": self.num_neighbors,
            "patch_size": self.patch_size,
            "patch_stride": self.patch_stride,
            "pretrain_embed_dimension": self.pretrain_embed_dimension,
            "target_embed_dimension": self.target_embed_dimension,
            "max_patches_per_image": self.max_patches_per_image,
            "max_training_embeddings": self.max_training_embeddings,
            "max_memory_bank_size": self.max_memory_bank_size,
            "projection_dim": self.projection_dim,
            "sampling_seed": self.sampling_seed,
            "calibration_quantile": self.calibration_quantile,
            "calibration_batches": self.calibration_batches,
            "distance_query_chunk_size": self.distance_query_chunk_size,
            "distance_bank_chunk_size": self.distance_bank_chunk_size,
            "gaussian_sigma": self.gaussian_sigma,
        }

    def train(self, mode: bool = True) -> PatchCore:
        """Keep the frozen backbone and its BatchNorm layers in evaluation mode."""
        super().train(False)
        return self

    @staticmethod
    def _validate_images(images: torch.Tensor) -> None:
        if images.ndim != 4 or images.shape[1] != 3:
            raise ValueError(f"Expected RGB images [B, 3, H, W], got {tuple(images.shape)}")
        if not images.is_floating_point() or not torch.isfinite(images).all():
            raise ValueError("images must contain finite floating-point values")

    def _patchify(
        self,
        feature: torch.Tensor,
    ) -> tuple[torch.Tensor, tuple[int, int]]:
        """Extract overlapping local patches without reducing their channels."""
        padding = self.patch_size // 2
        patches = F.unfold(
            feature,
            kernel_size=self.patch_size,
            stride=self.patch_stride,
            padding=padding,
        ).transpose(1, 2)
        height = (feature.shape[-2] + 2 * padding - self.patch_size) // self.patch_stride + 1
        width = (feature.shape[-1] + 2 * padding - self.patch_size) // self.patch_stride + 1
        patches = patches.reshape(
            feature.shape[0],
            height * width,
            feature.shape[1],
            self.patch_size,
            self.patch_size,
        )
        return patches, (height, width)

    @staticmethod
    def _align_patch_grid(
        patches: torch.Tensor,
        source_size: tuple[int, int],
        target_size: tuple[int, int],
    ) -> torch.Tensor:
        """Interpolate patch grids while preserving channel and patch axes."""
        if source_size == target_size:
            return patches
        batch_size, _, channels, patch_height, patch_width = patches.shape
        patches = patches.reshape(
            batch_size,
            *source_size,
            channels,
            patch_height,
            patch_width,
        ).permute(0, 3, 4, 5, 1, 2)
        patches = patches.reshape(-1, 1, *source_size)
        patches = F.interpolate(
            patches,
            size=target_size,
            mode="bilinear",
            align_corners=False,
        )
        return patches.reshape(
            batch_size,
            channels,
            patch_height,
            patch_width,
            *target_size,
        ).permute(0, 4, 5, 1, 2, 3).reshape(
            batch_size,
            target_size[0] * target_size[1],
            channels,
            patch_height,
            patch_width,
        )

    @staticmethod
    def _map_embedding_dimension(
        patches: torch.Tensor,
        output_dimension: int,
    ) -> torch.Tensor:
        """Apply the reference MeanMapper operation to every local patch."""
        batch_size, num_patches = patches.shape[:2]
        flattened = patches.reshape(batch_size * num_patches, 1, -1)
        mapped = F.adaptive_avg_pool1d(flattened, output_dimension)
        return mapped.reshape(batch_size, num_patches, output_dimension)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """Return reference-style locally aggregated embeddings [B, D, h, w]."""
        self._validate_images(images)
        features = self.feature_extractor(images)
        patch_layers = [self._patchify(features[layer]) for layer in self.layers]
        target_size = patch_layers[0][1]
        mapped_layers = []
        for patches, patch_grid in patch_layers:
            patches = self._align_patch_grid(patches, patch_grid, target_size)
            mapped_layers.append(
                self._map_embedding_dimension(
                    patches,
                    self.pretrain_embed_dimension,
                )
            )
        embeddings = torch.cat(mapped_layers, dim=-1)
        embeddings = self._map_embedding_dimension(
            embeddings,
            self.target_embed_dimension,
        )
        return embeddings.transpose(1, 2).reshape(
            images.shape[0],
            self.target_embed_dimension,
            *target_size,
        )

    def _patch_embeddings(
        self,
        images: torch.Tensor,
    ) -> tuple[torch.Tensor, tuple[int, int]]:
        embedding_map = self(images)
        height, width = embedding_map.shape[-2:]
        embeddings = embedding_map.permute(0, 2, 3, 1).flatten(1, 2).contiguous()
        return embeddings, (height, width)

    def _sample_image_patches(
        self,
        embeddings: torch.Tensor,
        generator: torch.Generator,
    ) -> torch.Tensor:
        batch_size, num_patches, embedding_dim = embeddings.shape
        selected_count = min(self.max_patches_per_image, num_patches)
        if selected_count == num_patches:
            return embeddings.reshape(-1, embedding_dim)
        random_keys = torch.rand((batch_size, num_patches), generator=generator)
        indices = random_keys.topk(selected_count, dim=1, largest=False).indices
        indices = indices.to(embeddings.device)
        return embeddings.gather(
            1,
            indices.unsqueeze(-1).expand(-1, -1, embedding_dim),
        ).reshape(-1, embedding_dim)

    @staticmethod
    def _reduce_priority_reservoir(
        embedding_chunks: list[torch.Tensor],
        priority_chunks: list[torch.Tensor],
        maximum_size: int,
    ) -> tuple[list[torch.Tensor], list[torch.Tensor], int]:
        embeddings = torch.cat(embedding_chunks)
        priorities = torch.cat(priority_chunks)
        if embeddings.shape[0] > maximum_size:
            indices = priorities.topk(maximum_size, largest=True, sorted=False).indices
            embeddings = embeddings[indices]
            priorities = priorities[indices]
        return [embeddings], [priorities], embeddings.shape[0]

    def _build_coreset(
        self,
        candidates: torch.Tensor,
        *,
        device: torch.device,
        generator: torch.Generator,
    ) -> torch.Tensor:
        target_size = max(1, math.ceil(candidates.shape[0] * self.coreset_sampling_ratio))
        target_size = min(target_size, self.max_memory_bank_size, candidates.shape[0])
        if target_size == candidates.shape[0]:
            return candidates.contiguous()

        projection_dim = min(self.projection_dim, candidates.shape[1])
        projection = torch.randn(
            candidates.shape[1],
            projection_dim,
            generator=generator,
        ) / math.sqrt(projection_dim)
        projected = candidates.to(device) @ projection.to(device)

        first_index = int(torch.randint(candidates.shape[0], (1,), generator=generator))
        selected = torch.empty(target_size, dtype=torch.long, device=device)
        selected_mask = torch.zeros(candidates.shape[0], dtype=torch.bool, device=device)
        minimum_distances = torch.full(
            (candidates.shape[0],),
            torch.inf,
            dtype=projected.dtype,
            device=device,
        )
        current_index = first_index
        for position in tqdm(range(target_size), desc="PatchCore coreset", leave=False):
            selected[position] = current_index
            selected_mask[current_index] = True
            if position + 1 == target_size:
                break
            center = projected[current_index]
            distances = (projected - center).square().sum(dim=1)
            minimum_distances = torch.minimum(minimum_distances, distances)
            minimum_distances[selected_mask] = -1
            current_index = int(minimum_distances.argmax())
        return candidates[selected.cpu()].contiguous()

    @staticmethod
    def _batch_images(batch: Mapping[str, Any], device: torch.device) -> torch.Tensor:
        if "image" not in batch:
            raise KeyError("Every training batch must contain 'image'")
        labels = batch.get("label")
        if labels is not None and torch.any(torch.as_tensor(labels) != 0):
            raise ValueError("PatchCore.fit accepts only clean training images")
        return batch["image"].to(device, non_blocking=True)

    @torch.no_grad()
    def fit(
        self,
        train_loader: Iterable[Mapping[str, Any]],
        *,
        device: torch.device | str,
    ) -> PatchCore:
        """Build the bounded coreset memory bank and clean-score calibration."""
        device = torch.device(device)
        self.to(device)
        self.train(False)
        self.memory_bank = torch.empty(0, 0, device=device)
        self.image_score_scale.zero_()
        self.anomaly_map_scale.zero_()
        self.fit_summary = {}
        generator = torch.Generator().manual_seed(self.sampling_seed)
        embedding_chunks: list[torch.Tensor] = []
        priority_chunks: list[torch.Tensor] = []
        buffered_count = 0
        num_images = 0
        sampled_embeddings = 0
        patch_grid: tuple[int, int] | None = None

        for batch in tqdm(train_loader, desc="PatchCore feature extraction"):
            images = self._batch_images(batch, device)
            embeddings, patch_grid = self._patch_embeddings(images)
            sampled = self._sample_image_patches(embeddings, generator).cpu()
            priorities = torch.rand(sampled.shape[0], generator=generator)
            embedding_chunks.append(sampled)
            priority_chunks.append(priorities)
            buffered_count += sampled.shape[0]
            sampled_embeddings += sampled.shape[0]
            num_images += images.shape[0]
            if buffered_count >= 2 * self.max_training_embeddings:
                embedding_chunks, priority_chunks, buffered_count = (
                    self._reduce_priority_reservoir(
                        embedding_chunks,
                        priority_chunks,
                        self.max_training_embeddings,
                    )
                )

        if not embedding_chunks or patch_grid is None:
            raise ValueError("train_loader produced no images")
        embedding_chunks, _, candidate_count = self._reduce_priority_reservoir(
            embedding_chunks,
            priority_chunks,
            self.max_training_embeddings,
        )
        candidates = embedding_chunks[0]
        self.memory_bank = self._build_coreset(
            candidates,
            device=device,
            generator=generator,
        ).to(device)

        image_scores: list[torch.Tensor] = []
        patch_scores: list[torch.Tensor] = []
        for batch_index, batch in enumerate(
            tqdm(train_loader, desc="PatchCore calibration", leave=False)
        ):
            if batch_index >= self.calibration_batches:
                break
            images = self._batch_images(batch, device)
            raw_image_scores, raw_patch_scores, _ = self._raw_scores(images)
            image_scores.append(raw_image_scores.cpu())
            patch_scores.append(raw_patch_scores.flatten().cpu())
        if not image_scores:
            raise ValueError("train_loader must be re-iterable for calibration")

        epsilon = torch.finfo(torch.float32).eps
        image_scale = torch.quantile(
            torch.cat(image_scores).float(), self.calibration_quantile
        ).clamp_min(epsilon)
        map_scale = torch.quantile(
            torch.cat(patch_scores).float(), self.calibration_quantile
        ).clamp_min(epsilon)
        self.image_score_scale.copy_(image_scale.to(device))
        self.anomaly_map_scale.copy_(map_scale.to(device))
        self.fit_summary = {
            "num_training_images": num_images,
            "sampled_embeddings": sampled_embeddings,
            "reservoir_embeddings": candidate_count,
            "memory_bank_embeddings": self.memory_bank.shape[0],
            "embedding_dimension": self.memory_bank.shape[1],
            "patch_grid": list(patch_grid),
            "image_score_scale": float(self.image_score_scale),
            "anomaly_map_scale": float(self.anomaly_map_scale),
        }
        return self

    def _nearest_memory(
        self,
        queries: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if self.memory_bank.numel() == 0:
            raise RuntimeError("PatchCore must be fitted before nearest-neighbour search")
        if queries.shape[1] != self.memory_bank.shape[1]:
            raise ValueError("Query and memory-bank embedding dimensions do not match")

        nearest_distances = torch.empty(queries.shape[0], device=queries.device)
        nearest_indices = torch.empty(
            queries.shape[0], dtype=torch.long, device=queries.device
        )
        for query_start in range(0, queries.shape[0], self.distance_query_chunk_size):
            query_end = min(query_start + self.distance_query_chunk_size, queries.shape[0])
            query_chunk = queries[query_start:query_end]
            best_distances = torch.full(
                (query_chunk.shape[0],), torch.inf, device=queries.device
            )
            best_indices = torch.zeros(
                query_chunk.shape[0], dtype=torch.long, device=queries.device
            )
            for bank_start in range(0, self.memory_bank.shape[0], self.distance_bank_chunk_size):
                bank_end = min(
                    bank_start + self.distance_bank_chunk_size,
                    self.memory_bank.shape[0],
                )
                distances = torch.cdist(query_chunk, self.memory_bank[bank_start:bank_end])
                block_distances, block_indices = distances.min(dim=1)
                improved = block_distances < best_distances
                best_distances[improved] = block_distances[improved]
                best_indices[improved] = block_indices[improved] + bank_start
            nearest_distances[query_start:query_end] = best_distances
            nearest_indices[query_start:query_end] = best_indices
        return nearest_distances, nearest_indices

    def _reweighted_image_scores(
        self,
        embeddings: torch.Tensor,
        patch_scores: torch.Tensor,
        nearest_indices: torch.Tensor,
    ) -> torch.Tensor:
        maximum_scores, maximum_locations = patch_scores.max(dim=1)
        if self.num_neighbors == 1 or self.memory_bank.shape[0] == 1:
            return maximum_scores

        scores = []
        num_support = min(self.num_neighbors, self.memory_bank.shape[0])
        for batch_index in range(embeddings.shape[0]):
            patch_index = maximum_locations[batch_index]
            query = embeddings[batch_index, patch_index]
            anchor_index = nearest_indices[batch_index, patch_index]
            anchor = self.memory_bank[anchor_index]
            anchor_distances = torch.linalg.vector_norm(self.memory_bank - anchor, dim=1)
            support_indices = anchor_distances.topk(
                num_support, largest=False
            ).indices
            support_distances = torch.linalg.vector_norm(
                self.memory_bank[support_indices] - query,
                dim=1,
            )
            anchor_position = (support_indices == anchor_index).nonzero(as_tuple=False)[0, 0]
            weight = 1.0 - torch.softmax(support_distances, dim=0)[anchor_position]
            scores.append(maximum_scores[batch_index] * weight)
        return torch.stack(scores)

    def _raw_scores(
        self,
        images: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, tuple[int, int]]:
        embeddings, patch_grid = self._patch_embeddings(images)
        batch_size, num_patches, embedding_dim = embeddings.shape
        distances, nearest_indices = self._nearest_memory(
            embeddings.reshape(-1, embedding_dim)
        )
        patch_scores = distances.reshape(batch_size, num_patches)
        nearest_indices = nearest_indices.reshape(batch_size, num_patches)
        image_scores = self._reweighted_image_scores(
            embeddings, patch_scores, nearest_indices
        )
        return image_scores, patch_scores, patch_grid

    @staticmethod
    def _normalize(raw_scores: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
        epsilon = torch.finfo(raw_scores.dtype).eps
        return raw_scores / (raw_scores + scale.clamp_min(epsilon))

    @torch.no_grad()
    def predict_with_raw(
        self,
        images: torch.Tensor,
    ) -> tuple[AnomalyPrediction, torch.Tensor, torch.Tensor]:
        if not self.is_fitted:
            raise RuntimeError("PatchCore must be fitted before predict")
        input_size = images.shape[-2:]
        raw_image_scores, raw_patch_scores, patch_grid = self._raw_scores(images)
        raw_map = raw_patch_scores.reshape(images.shape[0], 1, *patch_grid)
        raw_map = F.interpolate(
            raw_map,
            size=input_size,
            mode="bilinear",
            align_corners=False,
        )
        if self.gaussian_sigma > 0:
            kernel_size = 2 * math.ceil(3 * self.gaussian_sigma) + 1
            raw_map = gaussian_blur(
                raw_map,
                [kernel_size, kernel_size],
                [self.gaussian_sigma, self.gaussian_sigma],
            )
        prediction = AnomalyPrediction(
            anomaly_score=self._normalize(raw_image_scores, self.image_score_scale),
            anomaly_map=self._normalize(raw_map, self.anomaly_map_scale),
        )
        return prediction, raw_image_scores, raw_map

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        prediction, _, _ = self.predict_with_raw(images)
        return prediction

    def _prepare_state_dict_for_load(self, state_dict: Mapping[str, torch.Tensor]) -> None:
        memory_bank = state_dict.get("memory_bank")
        if memory_bank is not None and memory_bank.shape != self.memory_bank.shape:
            self.memory_bank = torch.empty_like(memory_bank)


import copy
import hashlib
import math
from collections.abc import Iterable, Mapping
from pathlib import Path
from typing import Any

import torch
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torchvision.models import get_model
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.transforms.functional import gaussian_blur, pil_to_tensor



def _capture_training_rng_state(train_loader) -> dict[str, Any]:
    generator = getattr(train_loader, "generator", None)
    return {
        "torch_rng_state": torch.get_rng_state(),
        "python_rng_state": random.getstate(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        ),
        "train_loader_generator_state": (
            generator.get_state() if generator is not None else None
        ),
    }


def _restore_training_rng_state(checkpoint: dict[str, Any], train_loader) -> None:
    generator_state = checkpoint.get("train_loader_generator_state")
    generator = getattr(train_loader, "generator", None)
    if generator_state is not None:
        if generator is None:
            raise TypeError("The train loader cannot restore its generator state")
        generator.set_state(
            torch.as_tensor(generator_state, dtype=torch.uint8, device="cpu")
        )
    torch.set_rng_state(
        torch.as_tensor(checkpoint["torch_rng_state"], dtype=torch.uint8, device="cpu")
    )
    if checkpoint.get("python_rng_state") is not None:
        random.setstate(checkpoint["python_rng_state"])
    cuda_rng_state = checkpoint.get("cuda_rng_state")
    if torch.cuda.is_available() and cuda_rng_state is not None:
        torch.cuda.set_rng_state_all([
            torch.as_tensor(state, dtype=torch.uint8, device="cpu")
            for state in cuda_rng_state
        ])


def _save_checkpoint_atomic(checkpoint: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(checkpoint, temporary)
    temporary.replace(path)


def _clean_images(batch: Mapping[str, Any], device: torch.device) -> torch.Tensor:
    if "image" not in batch:
        raise KeyError("Every training batch must contain 'image'")
    if torch.any(torch.as_tensor(batch.get("label", 0)) != 0):
        raise ValueError("fit accepts only clean training images")
    return batch["image"].to(device, non_blocking=True)


def _image_auc(scores: list[float], labels: list[int]) -> float:
    positives = [score for score, label in zip(scores, labels) if label == 1]
    negatives = [score for score, label in zip(scores, labels) if label == 0]
    if not positives or not negatives:
        return float("nan")
    wins = sum(p > n for p in positives for n in negatives)
    ties = sum(p == n for p in positives for n in negatives)
    return (wins + 0.5 * ties) / (len(positives) * len(negatives))


@torch.no_grad()
def _validation_auc(
    model, loader, device: torch.device, max_batches: int | None,
) -> float:
    if loader is None:
        return float("nan")
    model.eval()
    scores: list[float] = []
    labels: list[int] = []
    for index, batch in enumerate(loader):
        if max_batches is not None and index >= max_batches:
            break
        prediction = model.predict(batch["image"].to(device, non_blocking=True))
        scores.extend(prediction.anomaly_score.cpu().tolist())
        labels.extend(torch.as_tensor(batch["label"]).int().tolist())
    return _image_auc(scores, labels)


class PDNSmall(nn.Module):
    """EfficientAD PDN-S, including the paper's internal ImageNet normalization."""

    def __init__(self, output_channels: int = 384) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(3, 128, kernel_size=4, padding=3)
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2, padding=1)
        self.conv2 = nn.Conv2d(128, 256, kernel_size=4, padding=3)
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2, padding=1)
        self.conv3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(256, output_channels, kernel_size=4)

    @staticmethod
    def normalize(images: torch.Tensor) -> torch.Tensor:
        mean = images.new_tensor((0.485, 0.456, 0.406)).view(1, 3, 1, 1)
        std = images.new_tensor((0.229, 0.224, 0.225)).view(1, 3, 1, 1)
        return (images - mean) / std

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        value = self.normalize(images)
        value = F.relu(self.conv1(value))
        value = self.pool1(value)
        value = F.relu(self.conv2(value))
        value = self.pool2(value)
        value = F.relu(self.conv3(value))
        return self.conv4(value)


class EfficientADAutoencoder(nn.Module):
    """EfficientAD autoencoder with decoder geometry scaled from 256 px."""

    def __init__(self, output_channels: int = 384, output_size: int = 64) -> None:
        super().__init__()
        if output_size < 1:
            raise ValueError("output_size must be positive")
        self.output_size = int(output_size)
        scale = self.output_size / 64
        self.decoder_sizes = tuple(
            max(1, round((size + 1) * scale) - 1)
            for size in (3, 8, 15, 32, 63, 127)
        )
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 8),
        )
        self.decoder_convs = nn.ModuleList([
            nn.Conv2d(64, 64, 4, padding=2),
            nn.Conv2d(64, 64, 4, padding=2),
            nn.Conv2d(64, 64, 4, padding=2),
            nn.Conv2d(64, 64, 4, padding=2),
            nn.Conv2d(64, 64, 4, padding=2),
            nn.Conv2d(64, 64, 4, padding=2),
        ])
        self.final_conv = nn.Conv2d(64, 64, 3, padding=1)
        self.output = nn.Conv2d(64, output_channels, 3, padding=1)
        self.dropout = nn.Dropout(0.2)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        value = self.encoder(PDNSmall.normalize(images))
        for size, convolution in zip(self.decoder_sizes, self.decoder_convs):
            value = F.interpolate(value, size=(size, size), mode="bilinear", align_corners=False)
            value = self.dropout(F.relu(convolution(value)))
        value = F.interpolate(
            value, size=(self.output_size, self.output_size),
            mode="bilinear", align_corners=False,
        )
        return self.output(F.relu(self.final_conv(value)))


class EfficientAD(AnomalyDetector):
    """EfficientAD-S; 256 is official geometry, larger inputs scale the AE."""

    _NELSON_TO_LOCAL = {
        "0.weight": "conv1.weight", "0.bias": "conv1.bias",
        "3.weight": "conv2.weight", "3.bias": "conv2.bias",
        "6.weight": "conv3.weight", "6.bias": "conv3.bias",
        "8.weight": "conv4.weight", "8.bias": "conv4.bias",
    }

    def __init__(
        self, *, input_size: tuple[int, int] = (256, 256),
        teacher_weights_path: str | None = None,
        require_teacher_weights: bool = True, channels: int = 384,
        max_steps: int = 70_000, learning_rate: float = 1e-4,
        weight_decay: float = 1e-5, hard_quantile: float = 0.995,
        checkpoint_interval: int = 5_000, validation_interval: int = 1_000,
        early_stopping_patience: int = 8, early_stopping_min_steps: int = 0,
        early_stopping_min_relative_improvement: float = 0.001,
        lr_scheduler_patience: int = 3,
        fixed_training_duration: bool = False, mixed_precision: bool = True,
        spatial_calibration_enabled: bool = False,
        spatial_calibration_q_low: float = 0.90,
        spatial_calibration_q_high: float = 0.995,
        spatial_calibration_smoothing_sigma: float = 1.0,
        spatial_scale_floor_fraction: float = 0.25,
        static_roi_min_coverage: float = 0.50,
        image_score_topk_candidates: tuple[float, ...] = (
            0.0, 0.0005, 0.001, 0.0025, 0.005,
        ),
    ) -> None:
        super().__init__()
        input_size = tuple(int(value) for value in input_size)
        if (
            len(input_size) != 2 or input_size[0] != input_size[1]
            or input_size[0] < 256 or input_size[0] % 128 != 0
        ):
            raise ValueError(
                "EfficientAD input_size must be square, at least 256, "
                "and divisible by 128"
            )
        if channels < 1:
            raise ValueError("channels must be at least 1")
        if max_steps < 1:
            raise ValueError("max_steps must be at least 1")
        if checkpoint_interval < 1:
            raise ValueError("checkpoint_interval must be at least 1")
        if validation_interval < 1:
            raise ValueError("validation_interval must be at least 1")
        if early_stopping_patience < 1:
            raise ValueError("early_stopping_patience must be at least 1")
        if not 0 <= early_stopping_min_steps <= max_steps:
            raise ValueError("early_stopping_min_steps must be in [0, max_steps]")
        if not 0 <= early_stopping_min_relative_improvement < 1:
            raise ValueError(
                "early_stopping_min_relative_improvement must be in [0, 1)"
            )
        if lr_scheduler_patience < 0:
            raise ValueError("lr_scheduler_patience cannot be negative")
        if not 0 < hard_quantile <= 1:
            raise ValueError("hard_quantile must be in (0, 1]")
        if not 0 <= spatial_calibration_q_low < spatial_calibration_q_high <= 1:
            raise ValueError("Invalid spatial calibration quantiles")
        if spatial_calibration_smoothing_sigma < 0:
            raise ValueError("Spatial smoothing sigma cannot be negative")
        if spatial_scale_floor_fraction <= 0:
            raise ValueError("Spatial scale floor must be positive")
        if not 0 < static_roi_min_coverage <= 1:
            raise ValueError("Static ROI coverage must be in (0, 1]")
        image_score_topk_candidates = tuple(map(float, image_score_topk_candidates))
        if (
            not image_score_topk_candidates
            or len(set(image_score_topk_candidates)) != len(image_score_topk_candidates)
            or any(value < 0 or value > 1 for value in image_score_topk_candidates)
        ):
            raise ValueError("Top-k candidates must be unique fractions in [0, 1]")
        native_map_size = input_size[0] // 4
        self.teacher = PDNSmall(channels)
        self.student = PDNSmall(2 * channels)
        self.autoencoder = EfficientADAutoencoder(channels, native_map_size)
        self.teacher.requires_grad_(False)
        self.input_size = input_size
        self.native_map_size = native_map_size
        self.channels = channels
        self.max_steps = max_steps
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.hard_quantile = hard_quantile
        self.checkpoint_interval = checkpoint_interval
        self.validation_interval = validation_interval
        self.early_stopping_patience = early_stopping_patience
        self.early_stopping_min_steps = early_stopping_min_steps
        self.early_stopping_min_relative_improvement = (
            early_stopping_min_relative_improvement
        )
        self.lr_scheduler_patience = lr_scheduler_patience
        self.fixed_training_duration = bool(fixed_training_duration)
        self.mixed_precision = bool(mixed_precision)
        self.spatial_calibration_enabled = bool(spatial_calibration_enabled)
        self.spatial_calibration_q_low = float(spatial_calibration_q_low)
        self.spatial_calibration_q_high = float(spatial_calibration_q_high)
        self.spatial_calibration_smoothing_sigma = float(spatial_calibration_smoothing_sigma)
        self.spatial_scale_floor_fraction = float(spatial_scale_floor_fraction)
        self.static_roi_min_coverage = float(static_roi_min_coverage)
        self.image_score_topk_candidates = image_score_topk_candidates
        self.teacher_source = None
        if teacher_weights_path:
            self._load_teacher(Path(teacher_weights_path))
        elif require_teacher_weights:
            raise FileNotFoundError(
                "EfficientAD-S requires the nelson1425 teacher_small.pth checkpoint"
            )
        self.register_buffer("teacher_mean", torch.zeros(1, channels, 1, 1))
        self.register_buffer("teacher_std", torch.ones(1, channels, 1, 1))
        self.register_buffer("map_st_q90", torch.tensor(0.0))
        self.register_buffer("map_st_q995", torch.tensor(1.0))
        self.register_buffer("map_ae_q90", torch.tensor(0.0))
        self.register_buffer("map_ae_q995", torch.tensor(1.0))
        native_shape = (1, 1, native_map_size, native_map_size)
        self.register_buffer("map_st_spatial_q_low", torch.zeros(native_shape))
        self.register_buffer("map_st_spatial_q_high", torch.ones(native_shape))
        self.register_buffer("map_ae_spatial_q_low", torch.zeros(native_shape))
        self.register_buffer("map_ae_spatial_q_high", torch.ones(native_shape))
        self.register_buffer("static_roi", torch.ones(native_shape, dtype=torch.bool))
        self.register_buffer("selected_spatial_calibration", torch.tensor(False))
        self.register_buffer("selected_topk_fraction", torch.tensor(0.0))
        self.register_buffer("fitted", torch.tensor(False))
        self.fit_summary: dict[str, Any] = {}
        # Transient diagnostics never enter state_dict/checkpoints.
        self._diagnostic_spatial_override = None
        self._diagnostic_static_roi_override = None
        self._diagnostic_topk_fraction_override = None
        self._diagnostic_global_topk_pixels_override = None

    def train(self, mode: bool = True):
        super().train(mode)
        self.teacher.eval()
        return self

    def _load_teacher(self, path: Path) -> None:
        if not path.is_file():
            raise FileNotFoundError(f"EfficientAD teacher weights are missing: {path}")
        try:
            checkpoint = torch.load(path, map_location="cpu", weights_only=True)
        except TypeError:
            checkpoint = torch.load(path, map_location="cpu")
        state = checkpoint.get("state_dict", checkpoint)
        if not isinstance(state, Mapping):
            raise TypeError("The nelson1425 teacher checkpoint is not a state dict")
        if set(state) == set(self._NELSON_TO_LOCAL):
            state = {self._NELSON_TO_LOCAL[key]: value for key, value in state.items()}
        expected = set(self.teacher.state_dict())
        if set(state) != expected:
            raise ValueError(
                "Teacher checkpoint is incompatible with nelson1425 PDN-S: "
                f"expected {sorted(expected)}, got {sorted(state)}"
            )
        self.teacher.load_state_dict(state, strict=True)
        self.teacher_source = str(path)

    @property
    def is_fitted(self) -> bool:
        return bool(self.fitted)

    def checkpoint_config(self) -> dict[str, Any]:
        return {
            "input_size": self.input_size,
            "channels": self.channels,
            "max_steps": self.max_steps,
            "learning_rate": self.learning_rate,
            "weight_decay": self.weight_decay,
            "hard_quantile": self.hard_quantile,
            "checkpoint_interval": self.checkpoint_interval,
            "validation_interval": self.validation_interval,
            "early_stopping_patience": self.early_stopping_patience,
            "early_stopping_min_steps": self.early_stopping_min_steps,
            "early_stopping_min_relative_improvement": (
                self.early_stopping_min_relative_improvement
            ),
            "lr_scheduler_patience": self.lr_scheduler_patience,
            "fixed_training_duration": self.fixed_training_duration,
            "mixed_precision": self.mixed_precision,
            "spatial_calibration_enabled": self.spatial_calibration_enabled,
            "spatial_calibration_q_low": self.spatial_calibration_q_low,
            "spatial_calibration_q_high": self.spatial_calibration_q_high,
            "spatial_calibration_smoothing_sigma": self.spatial_calibration_smoothing_sigma,
            "spatial_scale_floor_fraction": self.spatial_scale_floor_fraction,
            "static_roi_min_coverage": self.static_roi_min_coverage,
            "image_score_topk_candidates": self.image_score_topk_candidates,
            "teacher_source": self.teacher_source,
        }

    def _validate_input_size(self, images: torch.Tensor, source: str) -> None:
        actual = tuple(images.shape[-2:])
        if actual != self.input_size:
            raise ValueError(
                f"EfficientAD expected {self.input_size} {source} images, got {actual}"
            )

    def _teacher(self, images: torch.Tensor) -> torch.Tensor:
        self._validate_input_size(images, "in-domain")
        value = self.teacher(images)
        return (value - self.teacher_mean) / self.teacher_std.clamp_min(1e-6)

    @staticmethod
    def _augment_autoencoder_input(images: torch.Tensor) -> torch.Tensor:
        factor = float(torch.empty(()).uniform_(0.8, 1.2))
        operation = int(torch.randint(3, ()).item())
        from torchvision.transforms import functional as transform_functional
        if operation == 0:
            return transform_functional.adjust_brightness(images, factor)
        if operation == 1:
            return transform_functional.adjust_contrast(images, factor)
        return transform_functional.adjust_saturation(images, factor)

    @torch.no_grad()
    def _raw_maps(self, images: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        teacher = self._teacher(images)
        student = self.student(images)
        autoencoder = self.autoencoder(images)
        map_st = (student[:, : self.channels] - teacher).square().mean(1, keepdim=True)
        map_ae = (
            student[:, self.channels :] - autoencoder
        ).square().mean(1, keepdim=True)
        return map_st, map_ae

    @torch.no_grad()
    def _clean_validation_loss(self, loader, device: torch.device) -> tuple[float, int]:
        """Evaluate only normal validation images with the in-domain train losses."""
        self.eval()
        total_loss = 0.0
        total_images = 0
        for batch in loader:
            labels = torch.as_tensor(batch.get("label", 0))
            clean = labels == 0
            if not clean.any():
                continue
            images = batch["image"][clean].to(device, non_blocking=True)
            teacher = self._teacher(images)
            student = self.student(images)
            distance = (student[:, : self.channels] - teacher).square()
            flattened = distance.flatten(1)
            thresholds = torch.quantile(
                flattened, self.hard_quantile, dim=1, keepdim=True
            )
            hard_mask = flattened >= thresholds
            hard_loss = (
                (flattened * hard_mask).sum(1)
                / hard_mask.sum(1).clamp_min(1)
            )
            autoencoder = self.autoencoder(images)
            student_autoencoder = student[:, self.channels :]
            autoencoder_loss = (
                (autoencoder - teacher).square().flatten(1).mean(1)
            )
            student_autoencoder_loss = (
                (student_autoencoder - autoencoder).square().flatten(1).mean(1)
            )
            per_image = hard_loss + autoencoder_loss + student_autoencoder_loss
            total_loss += float(per_image.sum())
            total_images += images.shape[0]
        if total_images == 0:
            raise ValueError(
                "early stopping requires at least one clean validation image"
            )
        return total_loss / total_images, total_images

    def _training_checkpoint(
        self, *, optimizer, scheduler, scaler, step: int, penalty_loader,
        train_epoch_generator_state=None, train_batches_into_epoch: int = 0,
        early_stopping_state: dict[str, Any] | None = None,
        optimization_complete: bool = False,
    ) -> dict[str, Any]:
        penalty_state = None
        penalty_dataset = getattr(penalty_loader, "dataset", None)
        if hasattr(penalty_dataset, "state_dict"):
            penalty_state = penalty_dataset.state_dict()
        return {
            "format": "efficientad_training_v4_spatial_calibration",
            "input_size": list(self.input_size),
            "model": self.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "grad_scaler": scaler.state_dict(),
            "step": step,
            "torch_rng_state": torch.get_rng_state(),
            "python_rng_state": random.getstate(),
            "train_epoch_generator_state": train_epoch_generator_state,
            "train_batches_into_epoch": train_batches_into_epoch,
            "cuda_rng_state": (
                torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
            ),
            "penalty_dataset_state": penalty_state,
            "early_stopping_state": early_stopping_state,
            "fixed_training_duration": self.fixed_training_duration,
            "mixed_precision": self.mixed_precision,
            "optimization_complete": optimization_complete,
        }

    @staticmethod
    def _save_training_checkpoint(checkpoint: dict[str, Any], path: Path) -> None:
        _save_checkpoint_atomic(checkpoint, path)

    def fit(
        self, train_loader, *, device, validation_loader=None,
        calibration_loader=None, penalty_loader=None, work_dir=None, resume=False,
    ) -> EfficientAD:
        if penalty_loader is None:
            raise ValueError("EfficientAD training requires an ImageNet penalty_loader")
        if validation_loader is None:
            raise ValueError("EfficientAD calibration requires a normal validation_loader")
        if self.spatial_calibration_enabled and calibration_loader is None:
            raise ValueError("Spatial calibration requires a disjoint clean calibration_loader")
        if self.max_steps < 1:
            raise ValueError("max_steps must be at least 1")
        device = torch.device(device)
        print(
            f"[EfficientAD] Starting fit: device={device}, "
            f"max_steps={self.max_steps:,}, resume={resume}, "
            f"training_mode={'fixed_duration' if self.fixed_training_duration else 'early_stopping'}, "
            f"validation_during_optimization={not self.fixed_training_duration}.",
            flush=True,
        )
        self.to(device)
        self.train(True)
        optimizer = torch.optim.Adam(
            [*self.student.parameters(), *self.autoencoder.parameters()],
            lr=self.learning_rate, weight_decay=self.weight_decay,
        )
        amp_enabled = self.mixed_precision and device.type == "cuda"
        try:
            scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
        except (AttributeError, TypeError):
            scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
        print(
            f"[EfficientAD] Mixed precision: "
            f"{'FP16 enabled' if amp_enabled else 'disabled'}",
            flush=True,
        )
        scheduler = (
            torch.optim.lr_scheduler.StepLR(
                optimizer, step_size=max(1, int(0.95 * self.max_steps)), gamma=0.1
            )
            if self.fixed_training_duration
            else torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="min", factor=0.1,
                patience=self.lr_scheduler_patience, min_lr=1e-6,
                threshold=self.early_stopping_min_relative_improvement,
                threshold_mode="rel",
            )
        )
        checkpoint_path = (
            None if work_dir is None else Path(work_dir) / "efficientad_training.ckpt"
        )
        step = 0
        resume_train_epoch_state = None
        resume_train_batches = 0
        checkpoint_rng_state = None
        checkpoint_python_rng_state = None
        checkpoint_cuda_rng_state = None
        best_state = None
        best_validation_loss = float("inf")
        best_step = 0
        checks_without_improvement = 0
        validation_history: list[dict[str, float | int]] = []
        optimization_complete = False
        stopped_early = False
        print(
            f"[EfficientAD] Training checkpoint: "
            f"{checkpoint_path if checkpoint_path is not None else 'disabled'}",
            flush=True,
        )
        if resume:
            if checkpoint_path is None or not checkpoint_path.is_file():
                raise FileNotFoundError(
                    "resume=true requires work_dir/efficientad_training.ckpt"
                )
            checkpoint = torch.load(
                checkpoint_path, map_location="cpu", weights_only=False
            )
            if checkpoint.get("format") != "efficientad_training_v4_spatial_calibration":
                raise RuntimeError(
                    "This run uses disjoint spatial calibration and pose crops; "
                    "start it with RESUME=False instead of loading a legacy checkpoint."
                )
            if bool(checkpoint.get("fixed_training_duration", False)) != (
                self.fixed_training_duration
            ):
                raise RuntimeError(
                    "The checkpoint training-duration mode differs from the current "
                    "configuration; start a new run with RESUME=False."
                )
            saved_input_size = tuple(checkpoint.get("input_size", (256, 256)))
            if saved_input_size != self.input_size:
                raise RuntimeError(
                    "The checkpoint input size differs from the current "
                    "configuration; start a new run with RESUME=False."
                )
            if bool(checkpoint.get("mixed_precision", False)) != self.mixed_precision:
                raise RuntimeError(
                    "The checkpoint mixed-precision mode differs from the current "
                    "configuration; start a new run with RESUME=False."
                )
            self.load_state_dict(checkpoint["model"])
            optimizer.load_state_dict(checkpoint["optimizer"])
            scheduler.load_state_dict(checkpoint["scheduler"])
            grad_scaler_state = checkpoint.get("grad_scaler")
            if grad_scaler_state:
                scaler.load_state_dict(grad_scaler_state)
            step = int(checkpoint["step"])
            early_state = checkpoint.get("early_stopping_state") or {}
            best_state = early_state.get("best_state")
            best_validation_loss = float(
                early_state.get("best_validation_loss", float("inf"))
            )
            best_step = int(early_state.get("best_step", 0))
            checks_without_improvement = int(
                early_state.get("checks_without_improvement", 0)
            )
            validation_history = list(early_state.get("validation_history", []))
            optimization_complete = bool(checkpoint.get("optimization_complete", False))
            stopped_early = bool(early_state.get("stopped_early", False))
            resume_train_epoch_state = checkpoint.get("train_epoch_generator_state")
            if resume_train_epoch_state is not None:
                resume_train_epoch_state = torch.as_tensor(
                    resume_train_epoch_state, dtype=torch.uint8, device="cpu"
                )
            resume_train_batches = int(checkpoint.get("train_batches_into_epoch", 0))
            if checkpoint.get("penalty_dataset_state") is not None:
                penalty_dataset = getattr(penalty_loader, "dataset", None)
                if not hasattr(penalty_dataset, "load_state_dict"):
                    raise TypeError("The penalty dataset cannot restore its stream state")
                penalty_dataset.load_state_dict(checkpoint["penalty_dataset_state"])
            checkpoint_rng_state = torch.as_tensor(
                checkpoint["torch_rng_state"], dtype=torch.uint8, device="cpu"
            )
            checkpoint_python_rng_state = checkpoint.get("python_rng_state")
            checkpoint_cuda_rng_state = checkpoint.get("cuda_rng_state")
            if checkpoint_cuda_rng_state is not None:
                checkpoint_cuda_rng_state = [
                    torch.as_tensor(state, dtype=torch.uint8, device="cpu")
                    for state in checkpoint_cuda_rng_state
                ]
            print(
                f"[EfficientAD] Resume state restored at step {step:,}; "
                f"train batches into epoch={resume_train_batches:,}; "
                f"optimization_complete={optimization_complete}.",
                flush=True,
            )

        if step == 0:
            print(
                "[EfficientAD] Computing teacher feature mean/std on the clean "
                "training set...",
                flush=True,
            )
            sums = torch.zeros(self.channels, device=device)
            squares = torch.zeros_like(sums)
            count = 0
            with torch.no_grad():
                for batch in tqdm(
                    train_loader, desc="EfficientAD teacher statistics",
                    unit="batch", leave=False,
                ):
                    value = self.teacher(_clean_images(batch, device))
                    sums += value.sum((0, 2, 3))
                    squares += value.square().sum((0, 2, 3))
                    count += value.shape[0] * value.shape[2] * value.shape[3]
            if count == 0:
                raise ValueError("train_loader produced no images")
            mean = sums / count
            variance = (squares / count - mean.square()).clamp_min(1e-6)
            self.teacher_mean.copy_(mean.view(1, -1, 1, 1))
            self.teacher_std.copy_(variance.sqrt().view(1, -1, 1, 1))
            print(
                f"[EfficientAD] Teacher statistics ready from {count:,} feature "
                "vectors.",
                flush=True,
            )
        else:
            print(
                "[EfficientAD] Reusing teacher statistics from the resume checkpoint.",
                flush=True,
            )

        train_generator = getattr(train_loader, "generator", None)
        if resume_train_epoch_state is not None and train_generator is not None:
            train_generator.set_state(resume_train_epoch_state)
        train_epoch_generator_state = (
            train_generator.get_state() if train_generator is not None else None
        )
        train_iterator = iter(train_loader)
        train_batches_into_epoch = 0
        if (
            resume_train_epoch_state is not None
            and step < self.max_steps
            and not optimization_complete
        ):
            for _ in range(resume_train_batches):
                try:
                    next(train_iterator)
                except StopIteration as error:
                    raise RuntimeError("Saved train-loader position is invalid") from error
                train_batches_into_epoch += 1
        elif resume_train_epoch_state is not None:
            train_batches_into_epoch = resume_train_batches
            print(
                "[EfficientAD] Optimization is already complete; skipping train-loader "
                "position replay and proceeding to calibration.",
                flush=True,
            )
        penalty_iterator = iter(penalty_loader)
        penalty_batch_seen = False
        print(
            "[EfficientAD] Local ImageNet penalty iterator created. The first "
            "next() call will fill its shuffle buffer from cached shards.",
            flush=True,
        )
        if checkpoint_rng_state is not None:
            torch.set_rng_state(checkpoint_rng_state)
            if checkpoint_python_rng_state is not None:
                random.setstate(checkpoint_python_rng_state)
            if torch.cuda.is_available() and checkpoint_cuda_rng_state is not None:
                torch.cuda.set_rng_state_all(checkpoint_cuda_rng_state)
        progress = tqdm(
            total=self.max_steps, initial=step, desc="EfficientAD-S training",
            unit="step", dynamic_ncols=True,
        )
        while step < self.max_steps and not optimization_complete:
            try:
                batch = next(train_iterator)
            except StopIteration:
                train_epoch_generator_state = (
                    train_generator.get_state() if train_generator is not None else None
                )
                train_batches_into_epoch = 0
                train_iterator = iter(train_loader)
                try:
                    batch = next(train_iterator)
                except StopIteration as error:
                    raise ValueError("train_loader produced no images") from error
            if not penalty_batch_seen:
                print(
                    "[EfficientAD] Waiting for the first ImageNet penalty batch...",
                    flush=True,
                )
            try:
                penalty_batch = next(penalty_iterator)
            except StopIteration:
                penalty_iterator = iter(penalty_loader)
                try:
                    penalty_batch = next(penalty_iterator)
                except StopIteration as error:
                    raise ValueError("penalty_loader produced no images") from error
            if not penalty_batch_seen:
                penalty_batch_seen = True
                print(
                    f"[EfficientAD] First ImageNet penalty batch ready with shape "
                    f"{tuple(penalty_batch['image'].shape)}. Training can proceed.",
                    flush=True,
                )

            train_batches_into_epoch += 1
            images = _clean_images(batch, device)
            penalty_images = penalty_batch["image"].to(device, non_blocking=True)
            self._validate_input_size(penalty_images, "ImageNet penalty")
            augmented = self._augment_autoencoder_input(images)
            train_batch_size = images.shape[0]
            penalty_batch_size = penalty_images.shape[0]
            with torch.no_grad(), torch.autocast(
                device_type=device.type, dtype=torch.float16, enabled=amp_enabled
            ):
                teacher_pair = self._teacher(torch.cat((images, augmented), dim=0))
                teacher, teacher_augmented = teacher_pair.split(
                    (train_batch_size, train_batch_size), dim=0
                )
            with torch.autocast(
                device_type=device.type, dtype=torch.float16, enabled=amp_enabled
            ):
                student_all = self.student(
                    torch.cat((images, augmented, penalty_images), dim=0)
                )
                student, student_augmented, student_penalty = student_all.split(
                    (train_batch_size, train_batch_size, penalty_batch_size), dim=0
                )
                autoencoder = self.autoencoder(augmented)

            distance = (
                student[:, : self.channels].float() - teacher.float()
            ).square()
            threshold = torch.quantile(distance.detach(), self.hard_quantile)
            loss_hard = distance[distance >= threshold].mean()
            loss_penalty = (
                student_penalty[:, : self.channels].float().square().mean()
            )
            student_autoencoder = student_augmented[:, self.channels :]
            loss_autoencoder = F.mse_loss(
                autoencoder.float(), teacher_augmented.float()
            )
            loss_student_autoencoder = F.mse_loss(
                student_autoencoder.float(), autoencoder.float()
            )
            loss = (
                loss_hard + loss_penalty + loss_autoencoder
                + loss_student_autoencoder
            )
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            step += 1
            if self.fixed_training_duration:
                scheduler.step()
            if step % 50 == 0 or step == self.max_steps:
                progress.set_postfix(
                    loss=f"{loss.detach().item():.4f}",
                    lr=f"{optimizer.param_groups[0]['lr']:.2e}", refresh=False,
                )
                progress.update(step - progress.n)

            should_stop = False
            if (
                not self.fixed_training_duration
                and (step % self.validation_interval == 0 or step == self.max_steps)
            ):
                validation_loss, validation_images = self._clean_validation_loss(
                    validation_loader, device
                )
                scheduler.step(validation_loss)
                improved = (
                    best_state is None
                    or validation_loss
                    < best_validation_loss
                    * (1.0 - self.early_stopping_min_relative_improvement)
                )
                if improved:
                    best_validation_loss = validation_loss
                    best_step = step
                    checks_without_improvement = 0
                    best_state = {
                        key: value.detach().cpu().clone()
                        for key, value in self.state_dict().items()
                    }
                else:
                    checks_without_improvement += 1
                validation_history.append({
                    "step": step,
                    "clean_validation_loss": validation_loss,
                    "clean_validation_images": validation_images,
                    "learning_rate": optimizer.param_groups[0]["lr"],
                    "improved": int(improved),
                })
                print(
                    f"[EfficientAD] Clean validation at step {step:,}: "
                    f"loss={validation_loss:.6g}, best={best_validation_loss:.6g} "
                    f"at step {best_step:,}, bad_checks="
                    f"{checks_without_improvement}/{self.early_stopping_patience}, "
                    f"lr={optimizer.param_groups[0]['lr']:.2e}.",
                    flush=True,
                )
                self.train(True)
                should_stop = (
                    step >= self.early_stopping_min_steps
                    and checks_without_improvement >= self.early_stopping_patience
                )
                if should_stop:
                    stopped_early = True
                    print(
                        f"[EfficientAD] Early stopping at step {step:,}; "
                        f"restoring best step {best_step:,}.",
                        flush=True,
                    )

            early_stopping_state = {
                "best_state": best_state,
                "best_validation_loss": best_validation_loss,
                "best_step": best_step,
                "checks_without_improvement": checks_without_improvement,
                "validation_history": validation_history,
                "stopped_early": stopped_early,
            }
            if (
                checkpoint_path is not None
                and (
                    step % self.checkpoint_interval == 0
                    or step == self.max_steps
                    or should_stop
                )
            ):
                self._save_training_checkpoint(
                    self._training_checkpoint(
                        optimizer=optimizer, scheduler=scheduler, scaler=scaler,
                        step=step,
                        penalty_loader=penalty_loader,
                        train_epoch_generator_state=train_epoch_generator_state,
                        train_batches_into_epoch=train_batches_into_epoch,
                        early_stopping_state=early_stopping_state,
                        optimization_complete=should_stop,
                    ),
                    checkpoint_path,
                )
                print(
                    f"[EfficientAD] Checkpoint saved at step {step:,}: "
                    f"{checkpoint_path}",
                    flush=True,
                )
            if should_stop:
                break

        if step > progress.n:
            progress.update(step - progress.n)
        progress.close()
        optimization_step = step
        if self.fixed_training_duration:
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in self.state_dict().items()
            }
            best_step = optimization_step
            print(
                f"[EfficientAD] Fixed-duration optimization complete at step "
                f"{best_step:,}; validation was not used for model selection.",
                flush=True,
            )
        else:
            if best_state is None:
                raise RuntimeError("EfficientAD completed without a validation checkpoint")
            self.load_state_dict(best_state)
            print(
                f"[EfficientAD] Selected clean-validation checkpoint from step "
                f"{best_step:,} (loss={best_validation_loss:.6g}).",
                flush=True,
            )
        print(
            "[EfficientAD] Optimization complete; starting clean calibration.",
            flush=True,
        )
        self.eval()
        calibration_source = calibration_loader if self.spatial_calibration_enabled else validation_loader
        st_maps: list[torch.Tensor] = []
        ae_maps: list[torch.Tensor] = []
        roi_masks: list[torch.Tensor] = []
        normal_images = 0
        with torch.no_grad():
            for batch in tqdm(
                calibration_source, desc="EfficientAD calibration",
                unit="batch", leave=False,
            ):
                labels = torch.as_tensor(batch.get("label", 0))
                clean = labels == 0
                if clean.any():
                    images = batch["image"][clean].to(device, non_blocking=True)
                    map_st, map_ae = self._raw_maps(images)
                    st_maps.append(map_st.cpu())
                    ae_maps.append(map_ae.cpu())
                    if self.spatial_calibration_enabled:
                        if "target_mask" not in batch:
                            raise KeyError("Spatial calibration requires target_mask")
                        target_mask = F.interpolate(
                            batch["target_mask"][clean].float(),
                            size=map_st.shape[-2:], mode="nearest",
                        )
                        roi_masks.append(target_mask.cpu() > 0)
                    normal_images += images.shape[0]
        if not st_maps:
            raise ValueError("Calibration requires at least one clean image")
        st_tensor = torch.cat(st_maps).float()
        ae_tensor = torch.cat(ae_maps).float()
        st_q90 = torch.quantile(st_tensor.flatten(), 0.90)
        st_q995 = torch.maximum(torch.quantile(st_tensor.flatten(), 0.995), st_q90 + 1e-6)
        ae_q90 = torch.quantile(ae_tensor.flatten(), 0.90)
        ae_q995 = torch.maximum(torch.quantile(ae_tensor.flatten(), 0.995), ae_q90 + 1e-6)
        self.map_st_q90.copy_(st_q90.to(device))
        self.map_st_q995.copy_(st_q995.to(device))
        self.map_ae_q90.copy_(ae_q90.to(device))
        self.map_ae_q995.copy_(ae_q995.to(device))
        if self.spatial_calibration_enabled:
            spatial = self._build_spatial_calibration(
                st_tensor, ae_tensor, torch.cat(roi_masks).float()
            )
            self.map_st_spatial_q_low.copy_(spatial["st_q_low"].to(device))
            self.map_st_spatial_q_high.copy_(spatial["st_q_high"].to(device))
            self.map_ae_spatial_q_low.copy_(spatial["ae_q_low"].to(device))
            self.map_ae_spatial_q_high.copy_(spatial["ae_q_high"].to(device))
            self.static_roi.copy_(spatial["roi"].to(device))
            scoring_ablation = self._select_validation_scoring(validation_loader, device)
        else:
            scoring_ablation = {
                "selection_split": "none", "selected": "global_max",
                "validation": {},
            }
        print(
            f"[EfficientAD] Calibration complete on {normal_images:,} normal images: "
            f"ST(q90={st_q90.item():.6g}, q99.5={st_q995.item():.6g}), "
            f"AE(q90={ae_q90.item():.6g}, q99.5={ae_q995.item():.6g}).",
            flush=True,
        )
        self.fitted.fill_(True)
        self.fit_summary = {
            "steps": optimization_step,
            "selected_step": best_step,
            "stopped_early": stopped_early,
            "fixed_training_duration": self.fixed_training_duration,
            "mixed_precision": self.mixed_precision,
            "best_clean_validation_loss": (
                None if self.fixed_training_duration else best_validation_loss
            ),
            "validation_interval": (
                None if self.fixed_training_duration else self.validation_interval
            ),
            "early_stopping_patience": (
                None if self.fixed_training_duration else self.early_stopping_patience
            ),
            "early_stopping_min_steps": (
                None if self.fixed_training_duration else self.early_stopping_min_steps
            ),
            "early_stopping_min_relative_improvement": (
                None if self.fixed_training_duration
                else self.early_stopping_min_relative_improvement
            ),
            "validation_history": validation_history,
            "normal_calibration_images": normal_images,
            "calibration_pixels_per_map": st_tensor.numel(),
            "spatial_calibration_enabled": self.spatial_calibration_enabled,
            "static_roi_fraction": float(self.static_roi.float().mean()),
            "image_score_ablation": scoring_ablation,
        }
        if checkpoint_path is not None:
            self._save_training_checkpoint(
                self._training_checkpoint(
                    optimizer=optimizer, scheduler=scheduler, scaler=scaler,
                    step=optimization_step, penalty_loader=penalty_loader,
                    train_epoch_generator_state=train_epoch_generator_state,
                    train_batches_into_epoch=train_batches_into_epoch,
                    early_stopping_state={
                        "best_state": best_state,
                        "best_validation_loss": best_validation_loss,
                        "best_step": best_step,
                        "checks_without_improvement": checks_without_improvement,
                        "validation_history": validation_history,
                        "stopped_early": stopped_early,
                    },
                    optimization_complete=True,
                ),
                checkpoint_path,
            )
        return self

    def _smooth_spatial_statistic(self, value):
        sigma = self.spatial_calibration_smoothing_sigma
        if sigma == 0:
            return value
        kernel = min(63, 2 * math.ceil(3 * sigma) + 1)
        if kernel % 2 == 0:
            kernel -= 1
        return gaussian_blur(value, [kernel, kernel], [sigma, sigma])

    def _build_spatial_calibration(self, st_tensor, ae_tensor, roi_masks):
        def statistics(values, global_low, global_high):
            low = torch.quantile(
                values, self.spatial_calibration_q_low, dim=0
            ).unsqueeze(0)
            high = torch.quantile(
                values, self.spatial_calibration_q_high, dim=0
            ).unsqueeze(0)
            low = self._smooth_spatial_statistic(low)
            high = self._smooth_spatial_statistic(high)
            scale_floor = (global_high - global_low).clamp_min(1e-6)
            scale_floor = scale_floor * self.spatial_scale_floor_fraction
            return low, low + (high - low).clamp_min(scale_floor)
        st_low, st_high = statistics(
            st_tensor, self.map_st_q90.cpu(), self.map_st_q995.cpu()
        )
        ae_low, ae_high = statistics(
            ae_tensor, self.map_ae_q90.cpu(), self.map_ae_q995.cpu()
        )
        roi = roi_masks.mean(dim=0, keepdim=True) >= self.static_roi_min_coverage
        if not roi.any():
            raise ValueError("Static ROI is empty; audit the clean target masks")
        return {
            "st_q_low": st_low, "st_q_high": st_high,
            "ae_q_low": ae_low, "ae_q_high": ae_high, "roi": roi,
        }

    @staticmethod
    def _calibrate_map(
        value: torch.Tensor, q90: torch.Tensor, q995: torch.Tensor,
    ) -> torch.Tensor:
        return 0.1 * (value - q90) / (q995 - q90).clamp_min(1e-6)

    def _combine_calibrated_maps(self, map_st, map_ae, *, spatial):
        if spatial:
            return 0.5 * (
                self._calibrate_map(
                    map_st, self.map_st_spatial_q_low, self.map_st_spatial_q_high
                )
                + self._calibrate_map(
                    map_ae, self.map_ae_spatial_q_low, self.map_ae_spatial_q_high
                )
            )
        return 0.5 * (
            self._calibrate_map(map_st, self.map_st_q90, self.map_st_q995)
            + self._calibrate_map(map_ae, self.map_ae_q90, self.map_ae_q995)
        )

    def _roi_at_size(self, size):
        return F.interpolate(self.static_roi.float(), size=size, mode="nearest").bool()

    def _aggregate_image_score(self, anomaly_map, *, use_static_roi, topk_fraction):
        if use_static_roi:
            roi = self._roi_at_size(anomaly_map.shape[-2:]).expand(
                anomaly_map.shape[0], -1, -1, -1
            )
            values = anomaly_map.masked_select(roi).view(anomaly_map.shape[0], -1)
        else:
            values = anomaly_map.flatten(1)
        if topk_fraction == 0:
            return values.amax(1)
        k = max(1, math.ceil(values.shape[1] * topk_fraction))
        return values.topk(k, dim=1).values.mean(1)

    @staticmethod
    def _topk_candidate_name(fraction):
        if fraction == 0:
            return "spatial_max"
        return f"spatial_topk_{fraction:g}".replace(".", "_")

    @torch.no_grad()
    def _select_validation_scoring(self, validation_loader, device):
        candidates = [("global_max", False, 0.0)] + [
            (self._topk_candidate_name(fraction), True, fraction)
            for fraction in self.image_score_topk_candidates
        ]
        scores = {name: [] for name, _, _ in candidates}
        labels = []
        self.eval()
        for batch in tqdm(
            validation_loader, desc="EfficientAD image-score ablation",
            unit="batch", leave=False,
        ):
            images = batch["image"].to(device, non_blocking=True)
            map_st, map_ae = self._raw_maps(images)
            native = {
                False: self._combine_calibrated_maps(map_st, map_ae, spatial=False),
                True: self._combine_calibrated_maps(
                    map_st, map_ae, spatial=True
                ) * self.static_roi,
            }
            resized = {
                spatial: F.interpolate(
                    value, images.shape[-2:], mode="bilinear", align_corners=False
                ) for spatial, value in native.items()
            }
            for name, spatial, fraction in candidates:
                value = self._aggregate_image_score(
                    resized[spatial], use_static_roi=spatial, topk_fraction=fraction
                )
                scores[name].extend(value.cpu().tolist())
            labels.extend(torch.as_tensor(batch["label"]).int().tolist())
        validation = {
            name: {"image_auroc": _image_auc(values, labels)}
            for name, values in scores.items()
        }
        if not labels or any(
            not math.isfinite(result["image_auroc"])
            for result in validation.values()
        ):
            raise ValueError("Scoring selection requires both validation classes")
        selected_name, selected_spatial, selected_fraction = max(
            candidates, key=lambda item: validation[item[0]]["image_auroc"]
        )
        self.selected_spatial_calibration.fill_(selected_spatial)
        self.selected_topk_fraction.fill_(selected_fraction)
        print(
            f"[EfficientAD] Validation selected {selected_name}: "
            f"image_AUROC={validation[selected_name]['image_auroc']:.6f}."
        )
        return {
            "selection_split": "validation",
            "selection_metric": "image_auroc",
            "selected": selected_name,
            "selected_spatial_calibration": selected_spatial,
            "selected_topk_fraction": selected_fraction,
            "validation": validation,
        }

    def configure_diagnostic_inference(self, mode):
        modes = {
            "global": (False, False),
            "global_roi": (False, True),
            "spatial_roi": (True, True),
        }
        if mode not in modes:
            raise ValueError(f"Unknown EfficientAD diagnostic mode: {mode}")
        spatial, use_static_roi = modes[mode]
        self._diagnostic_spatial_override = spatial
        self._diagnostic_static_roi_override = use_static_roi
        self._diagnostic_topk_fraction_override = 0.0
        self._diagnostic_global_topk_pixels_override = None

    def configure_diagnostic_global_topk_pixels(self, topk_pixels):
        if topk_pixels is not None and topk_pixels < 1:
            raise ValueError("topk_pixels must be a positive integer or None")
        self._diagnostic_global_topk_pixels_override = topk_pixels

    def clear_diagnostic_inference(self):
        self._diagnostic_spatial_override = None
        self._diagnostic_static_roi_override = None
        self._diagnostic_topk_fraction_override = None
        self._diagnostic_global_topk_pixels_override = None

    @torch.no_grad()
    def predict_with_raw(self, images: torch.Tensor):
        if not self.is_fitted:
            raise RuntimeError("EfficientAD must be fitted before predict")
        map_st, map_ae = self._raw_maps(images)
        spatial = (
            bool(self.selected_spatial_calibration)
            if self._diagnostic_spatial_override is None
            else self._diagnostic_spatial_override
        )
        use_static_roi = (
            spatial if self._diagnostic_static_roi_override is None
            else self._diagnostic_static_roi_override
        )
        topk_fraction = (
            float(self.selected_topk_fraction)
            if self._diagnostic_topk_fraction_override is None
            else self._diagnostic_topk_fraction_override
        )
        anomaly_map = self._combine_calibrated_maps(map_st, map_ae, spatial=spatial)
        if use_static_roi:
            anomaly_map = anomaly_map * self.static_roi
        raw_map = F.interpolate(
            anomaly_map, images.shape[-2:], mode="bilinear", align_corners=False
        )
        normalized_map = 0.5 + torch.atan(10.0 * raw_map) / math.pi
        if use_static_roi:
            normalized_map = normalized_map * self._roi_at_size(images.shape[-2:])
        native_topk_pixels = self._diagnostic_global_topk_pixels_override
        if native_topk_pixels is None:
            raw_score = self._aggregate_image_score(
                raw_map, use_static_roi=use_static_roi, topk_fraction=topk_fraction,
            )
        else:
            if spatial or use_static_roi:
                raise RuntimeError(
                    "Native global top-k requires global calibration without ROI"
                )
            native_values = anomaly_map.flatten(1)
            if native_topk_pixels > native_values.shape[1]:
                raise ValueError("topk_pixels exceeds native anomaly-map pixels")
            raw_score = native_values.topk(
                native_topk_pixels, dim=1
            ).values.mean(1)
        normalized_score = 0.5 + torch.atan(10.0 * raw_score) / math.pi
        return AnomalyPrediction(normalized_score, normalized_map), raw_score, raw_map

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        return self.predict_with_raw(images)[0]

def build_explicit_torchvision_backbone(
    backbone: str, *, pretrained: bool, weights_name: str,
) -> nn.Module:
    weights = None
    if pretrained:
        weights_enum = get_model_weights(backbone)
        try:
            weights = getattr(weights_enum, weights_name)
        except AttributeError as error:
            raise ValueError(
                f"Unknown torchvision weights {weights_name!r} for {backbone!r}"
            ) from error
    return get_model(backbone, weights=weights)


def _init_supersimplenet_weights(module: nn.Module) -> None:
    if isinstance(module, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_normal_(module.weight)
    elif isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d)):
        nn.init.constant_(module.weight, 1)


def _rand_perlin_2d(
    shape: tuple[int, int], res: tuple[int, int], *, device: torch.device,
) -> torch.Tensor:
    """Torch port of the Perlin generator used by official SuperSimpleNet."""
    delta = (res[0] / shape[0], res[1] / shape[1])
    repeats = (shape[0] // res[0], shape[1] // res[1])
    y = torch.arange(0, res[0], delta[0], device=device)
    x = torch.arange(0, res[1], delta[1], device=device)
    grid = torch.stack(torch.meshgrid(y, x, indexing="ij"), dim=-1) % 1
    angles = 2 * math.pi * torch.rand(res[0] + 1, res[1] + 1, device=device)
    gradients = torch.stack((torch.cos(angles), torch.sin(angles)), dim=-1)

    def tiled(y_slice, x_slice):
        return (
            gradients[y_slice[0]:y_slice[1], x_slice[0]:x_slice[1]]
            .repeat_interleave(repeats[0], 0)
            .repeat_interleave(repeats[1], 1)
        )

    def dot(gradient, shift):
        offsets = torch.stack(
            (grid[:shape[0], :shape[1], 0] + shift[0],
             grid[:shape[0], :shape[1], 1] + shift[1]),
            dim=-1,
        )
        return (offsets * gradient[:shape[0], :shape[1]]).sum(dim=-1)

    n00 = dot(tiled((0, -1), (0, -1)), (0, 0))
    n10 = dot(tiled((1, None), (0, -1)), (-1, 0))
    n01 = dot(tiled((0, -1), (1, None)), (0, -1))
    n11 = dot(tiled((1, None), (1, None)), (-1, -1))
    fade = lambda value: 6 * value**5 - 15 * value**4 + 10 * value**3
    blend = fade(grid[:shape[0], :shape[1]])
    return math.sqrt(2) * torch.lerp(
        torch.lerp(n00, n10, blend[..., 0]),
        torch.lerp(n01, n11, blend[..., 0]),
        blend[..., 1],
    )


def _ssn_focal_loss(
    probabilities: torch.Tensor, targets: torch.Tensor, gamma: float = 4.0,
    reduction: str | None = "mean",
) -> torch.Tensor:
    probabilities = probabilities.float()
    targets = targets.float()
    ce = F.binary_cross_entropy(probabilities, targets, reduction="none")
    pt = probabilities * targets + (1 - probabilities) * (1 - targets)
    loss = ce * (1 - pt).pow(gamma)
    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    return loss


class SuperSimpleFeatureExtractor(nn.Module):
    def __init__(
        self, *, backbone: str, pretrained: bool, weights_name: str,
        layers: tuple[str, ...], patch_size: int, input_size: tuple[int, int],
    ) -> None:
        super().__init__()
        model = build_explicit_torchvision_backbone(
            backbone, pretrained=pretrained, weights_name=weights_name
        )
        self.extractor = create_feature_extractor(
            model, return_nodes={layer: layer for layer in layers}
        )
        self.extractor.requires_grad_(False)
        self.layers = layers
        self.pooler = nn.AvgPool2d(patch_size, stride=1, padding=patch_size // 2)
        with torch.no_grad():
            outputs = self.extractor(torch.zeros(1, 3, *input_size))
        self.feature_channels = sum(value.shape[1] for value in outputs.values())

    def train(self, mode: bool = True):
        super().train(False)
        return self

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        self.extractor.eval()
        with torch.no_grad():
            features = list(self.extractor(images).values())
        height, width = features[0].shape[-2:]
        features = [
            F.interpolate(
                feature, size=(height * 2, width * 2),
                mode="bilinear", align_corners=True,
            )
            for feature in features
        ]
        return self.pooler(torch.cat(features, dim=1))


class SuperSimpleFeatureAdaptor(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.projection = nn.Conv2d(channels, channels, 1)
        self.apply(_init_supersimplenet_weights)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.projection(features)


class SuperSimpleDiscriminator(nn.Module):
    def __init__(self, channels: int, *, hidden_channels: int, stop_grad: bool) -> None:
        super().__init__()
        self.stop_grad = stop_grad
        self.segmentor = nn.Sequential(
            nn.Conv2d(channels, hidden_channels, 1),
            nn.BatchNorm2d(hidden_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(hidden_channels, 1, 1, bias=False),
        )
        self.decision_features = nn.Sequential(
            nn.Conv2d(channels + 1, 128, 5, padding="same"),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.score = nn.Linear(128 * 2 + 2, 1)
        self.apply(_init_supersimplenet_weights)

    def parameter_groups(self):
        return (
            self.segmentor.parameters(),
            [*self.decision_features.parameters(), *self.score.parameters()],
        )

    def forward(
        self, segmentation_features: torch.Tensor,
        classification_features: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        anomaly_map = self.segmentor(segmentation_features)
        decision_map = anomaly_map.detach() if self.stop_grad else anomaly_map
        decision = self.decision_features(
            torch.cat((classification_features, decision_map), dim=1)
        )
        map_for_pooling = decision_map if self.stop_grad else anomaly_map
        pooled = torch.cat((
            F.adaptive_max_pool2d(decision, 1),
            F.adaptive_avg_pool2d(decision, 1),
            F.adaptive_max_pool2d(map_for_pooling, 1),
            F.adaptive_avg_pool2d(map_for_pooling, 1),
        ), dim=1).flatten(1)
        return anomaly_map, self.score(pooled).flatten()


class SuperSimpleAnomalyGenerator(nn.Module):
    def __init__(
        self, *, noise_std: float, threshold: float,
        perlin_range: tuple[int, int] = (0, 6),
    ) -> None:
        super().__init__()
        self.noise_std = noise_std
        self.threshold = threshold
        self.perlin_range = perlin_range

    def _masks(
        self, batch_size: int, height: int, width: int, device: torch.device,
    ) -> torch.Tensor:
        power_height = 1 << (height - 1).bit_length()
        power_width = 1 << (width - 1).bit_length()
        masks = []
        for _ in range(batch_size):
            max_y = min(self.perlin_range[1], int(math.log2(power_height)) + 1)
            max_x = min(self.perlin_range[1], int(math.log2(power_width)) + 1)
            scale_y = 2 ** int(torch.randint(self.perlin_range[0], max_y, (1,)).item())
            scale_x = 2 ** int(torch.randint(self.perlin_range[0], max_x, (1,)).item())
            noise = _rand_perlin_2d(
                (power_height, power_width), (scale_y, scale_x), device=device
            )
            noise = F.interpolate(
                noise[None, None], size=(height, width),
                mode="bilinear", align_corners=False,
            )[0]
            mask = (noise > self.threshold).float()
            if torch.rand(()) > 0.5:
                mask.zero_()  # official no_anomaly="empty"
            masks.append(mask)
        return torch.stack(masks)

    def forward(
        self, features: torch.Tensor, adapted: torch.Tensor,
        target_masks: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        batch, _, height, width = adapted.shape
        features = torch.cat((features, features), dim=0)
        adapted = torch.cat((adapted, adapted), dim=0)
        mask = self._masks(batch * 2, height, width, adapted.device)
        if target_masks is not None:
            expected_shape = (batch, 1, height, width)
            if target_masks.shape != expected_shape:
                raise ValueError(
                    "SuperSimpleNet target masks must match the feature grid: "
                    f"expected {expected_shape}, got {tuple(target_masks.shape)}"
                )
            target_masks = (target_masks > 0).to(mask.dtype)
            if not target_masks.flatten(1).any(1).all():
                raise ValueError(
                    "SuperSimpleNet target masks must contain the target wheel"
                )
            mask = mask * torch.cat((target_masks, target_masks), dim=0)
        noise = torch.normal(0, self.noise_std, size=adapted.shape, device=adapted.device)
        return features + noise * mask, adapted + noise * mask, mask


class SuperSimpleNet(AnomalyDetector):
    """JIMS SuperSimpleNet in the official unsupervised MVTec configuration."""

    def __init__(
        self, *, backbone: str = "wide_resnet50_2",
        pretrained: bool = True, weights_name: str = "IMAGENET1K_V1",
        layers: tuple[str, ...] = ("layer2", "layer3"),
        input_size: tuple[int, int] = (256, 256), patch_size: int = 3,
        epochs: int = 300, noise_std: float = 0.015,
        perlin_threshold: float = 0.2, adaptor_learning_rate: float = 1e-4,
        segmentation_learning_rate: float = 2e-4,
        decision_learning_rate: float = 2e-4, scheduler_gamma: float = 0.4,
        stop_grad: bool = True, adapt_classification_features: bool = False,
        gradient_clip: bool = False, margin: float = 0.5,
        gaussian_sigma: float = 4.0, fixed_training_duration: bool = True,
        image_score_mode: str = "classification_head",
        image_score_fraction: float = 0.01,
        validation_interval: int = 4, validation_batches: int | None = 64,
        early_stopping_patience: int | None = 5,
        early_stopping_min_delta: float = 0.001,
        restrict_synthetic_anomalies_to_target_mask: bool = False,
        max_samples_per_epoch: int | None = None, checkpoint_interval: int = 1,
    ) -> None:
        super().__init__()
        self.backbone_name = backbone
        self.pretrained = pretrained
        self.weights_name = weights_name
        self.layers = tuple(layers)
        self.input_size = tuple(input_size)
        self.patch_size = patch_size
        self.epochs = epochs
        self.noise_std = noise_std
        self.perlin_threshold = perlin_threshold
        self.adaptor_learning_rate = adaptor_learning_rate
        self.segmentation_learning_rate = segmentation_learning_rate
        self.decision_learning_rate = decision_learning_rate
        self.scheduler_gamma = scheduler_gamma
        self.stop_grad = stop_grad
        self.adapt_classification_features = adapt_classification_features
        self.gradient_clip = gradient_clip
        self.margin = margin
        self.gaussian_sigma = gaussian_sigma
        if image_score_mode not in {"classification_head", "topk_fraction_mean"}:
            raise ValueError(
                "image_score_mode must be classification_head or "
                "topk_fraction_mean"
            )
        if not 0 < image_score_fraction <= 1:
            raise ValueError("image_score_fraction must be in (0, 1]")
        self.image_score_mode = image_score_mode
        self.image_score_fraction = float(image_score_fraction)
        self.fixed_training_duration = fixed_training_duration
        if validation_interval < 1:
            raise ValueError("validation_interval must be at least 1")
        self.validation_interval = validation_interval
        self.validation_batches = validation_batches
        if early_stopping_patience is not None and early_stopping_patience < 1:
            raise ValueError("early_stopping_patience must be positive or None")
        if early_stopping_min_delta < 0:
            raise ValueError("early_stopping_min_delta cannot be negative")
        self.early_stopping_patience = early_stopping_patience
        self.early_stopping_min_delta = early_stopping_min_delta
        self.restrict_synthetic_anomalies_to_target_mask = bool(
            restrict_synthetic_anomalies_to_target_mask
        )
        self.max_samples_per_epoch = max_samples_per_epoch
        if checkpoint_interval < 1:
            raise ValueError("checkpoint_interval must be at least 1")
        self.checkpoint_interval = checkpoint_interval

        self.features = SuperSimpleFeatureExtractor(
            backbone=backbone, pretrained=pretrained, weights_name=weights_name,
            layers=self.layers, patch_size=patch_size, input_size=self.input_size,
        )
        channels = self.features.feature_channels
        self.adaptor = SuperSimpleFeatureAdaptor(channels)
        self.discriminator = SuperSimpleDiscriminator(
            channels, hidden_channels=1024, stop_grad=stop_grad
        )
        self.anomaly_generator = SuperSimpleAnomalyGenerator(
            noise_std=noise_std, threshold=perlin_threshold
        )
        self.register_buffer("fitted", torch.tensor(False))
        self.fit_summary: dict[str, Any] = {}

    @property
    def is_fitted(self) -> bool:
        return bool(self.fitted)

    def train(self, mode: bool = True):
        super().train(mode)
        self.features.eval()
        return self

    def checkpoint_config(self) -> dict[str, Any]:
        return {
            "backbone": self.backbone_name, "pretrained": self.pretrained,
            "weights_name": self.weights_name, "layers": list(self.layers),
            "input_size": list(self.input_size), "patch_size": self.patch_size,
            "epochs": self.epochs, "noise_std": self.noise_std,
            "perlin_threshold": self.perlin_threshold,
            "adaptor_learning_rate": self.adaptor_learning_rate,
            "segmentation_learning_rate": self.segmentation_learning_rate,
            "decision_learning_rate": self.decision_learning_rate,
            "scheduler_gamma": self.scheduler_gamma, "stop_grad": self.stop_grad,
            "adapt_classification_features": self.adapt_classification_features,
            "gradient_clip": self.gradient_clip, "margin": self.margin,
            "gaussian_sigma": self.gaussian_sigma,
            "image_score_mode": self.image_score_mode,
            "image_score_fraction": self.image_score_fraction,
            "fixed_training_duration": self.fixed_training_duration,
            "validation_interval": self.validation_interval,
            "validation_batches": self.validation_batches,
            "early_stopping_patience": self.early_stopping_patience,
            "early_stopping_min_delta": self.early_stopping_min_delta,
            "restrict_synthetic_anomalies_to_target_mask": (
                self.restrict_synthetic_anomalies_to_target_mask
            ),
            "max_samples_per_epoch": self.max_samples_per_epoch,
        }

    def _training_state(self) -> dict[str, Any]:
        return {
            "adaptor": self.adaptor.state_dict(),
            "discriminator": self.discriminator.state_dict(),
        }

    def _load_training_state(self, state: dict[str, Any]) -> None:
        self.adaptor.load_state_dict(state["adaptor"])
        self.discriminator.load_state_dict(state["discriminator"])

    def _logits(self, images: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        features = self.features(images)
        adapted = self.adaptor(features)
        classification = adapted if self.adapt_classification_features else features
        return self.discriminator(adapted, classification)

    def _image_score_components(
        self, images: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        return self._logits(images)

    def fit(
        self, train_loader, *, device, validation_loader=None,
        work_dir=None, resume=False,
    ) -> SuperSimpleNet:
        device = torch.device(device)
        if not self.fixed_training_duration and validation_loader is None:
            raise ValueError(
                "SuperSimpleNet early stopping requires validation_loader"
            )
        self.to(device)
        segmentation_parameters, decision_parameters = self.discriminator.parameter_groups()
        optimizer = torch.optim.AdamW([
            {"params": self.adaptor.parameters(), "lr": self.adaptor_learning_rate},
            {"params": segmentation_parameters, "lr": self.segmentation_learning_rate,
             "weight_decay": 1e-5},
            {"params": decision_parameters, "lr": self.decision_learning_rate,
             "weight_decay": 1e-5},
        ])
        scheduler = torch.optim.lr_scheduler.MultiStepLR(
            optimizer,
            milestones=[int(self.epochs * 0.8), int(self.epochs * 0.9)],
            gamma=self.scheduler_gamma,
        )
        best_auc = -math.inf
        best_state = None
        best_epoch = None
        validation_history: list[dict[str, Any]] = []
        no_improvement_validations = 0
        stopped_early = False
        steps = 0
        start_epoch = 0
        completed_epoch = 0
        checkpoint_path = (
            None if work_dir is None
            else Path(work_dir) / "supersimplenet_training.ckpt"
        )
        if resume:
            if checkpoint_path is None or not checkpoint_path.is_file():
                raise FileNotFoundError(
                    "resume=true requires work_dir/supersimplenet_training.ckpt"
                )
            checkpoint = torch.load(
                checkpoint_path, map_location="cpu", weights_only=False
            )
            if checkpoint.get("format") != "supersimplenet_training_v1":
                raise ValueError("Unsupported SuperSimpleNet training checkpoint")
            if checkpoint.get("model_config") != self.checkpoint_config():
                raise ValueError("SuperSimpleNet training checkpoint config mismatch")
            self._load_training_state(checkpoint["model"])
            optimizer.load_state_dict(checkpoint["optimizer"])
            scheduler.load_state_dict(checkpoint["scheduler"])
            start_epoch = int(checkpoint["epoch"])
            completed_epoch = start_epoch
            if not 0 <= start_epoch <= self.epochs:
                raise ValueError("Invalid epoch in SuperSimpleNet training checkpoint")
            steps = int(checkpoint.get("steps", 0))
            best_auc = float(checkpoint.get("best_auc", -math.inf))
            best_state = checkpoint.get("best_state")
            best_epoch = checkpoint.get("best_epoch")
            validation_history = list(checkpoint.get("validation_history", []))
            no_improvement_validations = int(
                checkpoint.get("no_improvement_validations", 0)
            )
            stopped_early = bool(checkpoint.get("stopped_early", False))
            _restore_training_rng_state(checkpoint, train_loader)
        remaining_epochs = (
            () if stopped_early else range(start_epoch + 1, self.epochs + 1)
        )
        epoch_progress = tqdm(
            remaining_epochs,
            initial=start_epoch,
            total=self.epochs,
            desc="SuperSimpleNet training",
            unit="epoch",
            dynamic_ncols=True,
            leave=True,
        )
        for epoch in epoch_progress:
            completed_epoch = epoch
            self.train(True)
            samples = 0
            produced = False
            for batch in train_loader:
                produced = True
                images = _clean_images(batch, device)
                with torch.no_grad():
                    features = self.features(images)
                adapted = self.adaptor(features)
                target_masks = None
                if self.restrict_synthetic_anomalies_to_target_mask:
                    if "target_mask" not in batch:
                        raise KeyError(
                            "restrict_synthetic_anomalies_to_target_mask=true "
                            "requires target_mask in every training batch"
                        )
                    target_masks = F.interpolate(
                        batch["target_mask"].to(
                            device, non_blocking=True, dtype=torch.float32
                        ),
                        size=adapted.shape[-2:],
                        mode="nearest",
                    )
                noisy_features, noisy_adapted, target_mask = self.anomaly_generator(
                    features, adapted, target_masks=target_masks
                )
                classification = (
                    noisy_adapted if self.adapt_classification_features
                    else noisy_features
                )
                anomaly_map, score = self.discriminator(noisy_adapted, classification)
                target_label = target_mask.flatten(1).amax(1)

                focal = _ssn_focal_loss(
                    torch.sigmoid(anomaly_map), target_mask, reduction=None
                )
                truncated = torch.zeros_like(anomaly_map)
                normal = target_mask == 0
                anomalous = target_mask > 0
                truncated[normal] = torch.clamp(anomaly_map[normal] + self.margin, min=0)
                truncated[anomalous] = torch.clamp(-anomaly_map[anomalous] + self.margin, min=0)
                good_loss = truncated[normal].mean() if normal.any() else 0.0
                bad_loss = truncated[anomalous].mean() if anomalous.any() else 0.0
                loss = good_loss + bad_loss + focal.mean()
                loss = loss + _ssn_focal_loss(torch.sigmoid(score), target_label)

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if self.gradient_clip:
                    torch.nn.utils.clip_grad_norm_(self.parameters(), 1.0)
                optimizer.step()
                steps += 1
                samples += images.shape[0]
                if self.max_samples_per_epoch and samples >= self.max_samples_per_epoch:
                    break
            if not produced:
                raise ValueError("train_loader produced no images")
            epoch_progress.set_postfix(
                loss=f"{loss.detach().item():.4f}",
                refresh=True,
            )
            scheduler.step()
            if (
                not self.fixed_training_duration
                and (epoch % self.validation_interval == 0 or epoch == self.epochs)
            ):
                self.fitted.fill_(True)
                auc = _validation_auc(
                    self, validation_loader, device, self.validation_batches
                )
                validation_history.append({
                    "epoch": epoch,
                    "image_auroc": auc,
                })
                epoch_progress.set_postfix(
                    val_auc=f"{auc:.4f}",
                    refresh=True,
                )
                if not math.isnan(auc):
                    if auc > best_auc + self.early_stopping_min_delta:
                        best_auc = auc
                        best_epoch = epoch
                        best_state = copy.deepcopy(self._training_state())
                        no_improvement_validations = 0
                    else:
                        no_improvement_validations += 1
                    if (
                        self.early_stopping_patience is not None
                        and no_improvement_validations
                        >= self.early_stopping_patience
                    ):
                        stopped_early = True
            if (
                checkpoint_path is not None
                and (
                    epoch % self.checkpoint_interval == 0
                    or epoch == self.epochs
                    or stopped_early
                )
            ):
                _save_checkpoint_atomic({
                    "format": "supersimplenet_training_v1",
                    "model_config": self.checkpoint_config(),
                    "model": self._training_state(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "epoch": epoch,
                    "steps": steps,
                    "best_auc": best_auc,
                    "best_state": best_state,
                    "best_epoch": best_epoch,
                    "validation_history": validation_history,
                    "no_improvement_validations": no_improvement_validations,
                    "stopped_early": stopped_early,
                    **_capture_training_rng_state(train_loader),
                }, checkpoint_path)
            if stopped_early:
                break
        if best_state is not None:
            self._load_training_state(best_state)
        self.fitted.fill_(True)
        self.eval()
        self.fit_summary = {
            "epochs": self.epochs, "steps": steps,
            "epochs_completed": completed_epoch,
            "fixed_training_duration": self.fixed_training_duration,
            "selected_validation_image_auroc": (
                None if self.fixed_training_duration or best_state is None else best_auc
            ),
            "selected_epoch": best_epoch,
            "stopped_early": stopped_early,
            "no_improvement_validations": no_improvement_validations,
            "validation_history": validation_history,
            "restrict_synthetic_anomalies_to_target_mask": (
                self.restrict_synthetic_anomalies_to_target_mask
            ),
        }
        if checkpoint_path is not None:
            _save_checkpoint_atomic({
                "format": "supersimplenet_training_v1",
                "model_config": self.checkpoint_config(),
                "model": self._training_state(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "epoch": completed_epoch,
                "steps": steps,
                "best_auc": best_auc,
                "best_state": best_state,
                "best_epoch": best_epoch,
                "validation_history": validation_history,
                "no_improvement_validations": no_improvement_validations,
                "stopped_early": stopped_early,
                **_capture_training_rng_state(train_loader),
            }, checkpoint_path)
        return self

    @torch.no_grad()
    def predict_with_raw(self, images: torch.Tensor):
        if not self.is_fitted:
            raise RuntimeError("SuperSimpleNet must be fitted before predict")
        self.eval()
        native_raw_map, classification_raw_score = self._logits(images)
        if self.image_score_mode == "classification_head":
            raw_score = classification_raw_score
        else:
            flattened = native_raw_map.flatten(1)
            topk = min(
                flattened.shape[1],
                max(1, math.ceil(self.image_score_fraction * flattened.shape[1])),
            )
            raw_score = flattened.topk(topk, dim=1).values.mean(dim=1)
        raw_map = native_raw_map
        raw_map = F.interpolate(
            raw_map, images.shape[-2:], mode="bilinear", align_corners=False
        )
        if self.gaussian_sigma > 0:
            kernel = 2 * math.ceil(3 * self.gaussian_sigma) + 1
            raw_map = gaussian_blur(
                raw_map, [kernel, kernel], [self.gaussian_sigma, self.gaussian_sigma]
            )
        anomaly_map = torch.sigmoid(raw_map)
        anomaly_score = torch.sigmoid(raw_score)
        prediction = AnomalyPrediction(anomaly_score, anomaly_map)
        return prediction, raw_score, raw_map

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        return self.predict_with_raw(images)[0]

TINYGLASS_TRAINING_CHECKPOINT_FORMAT = "tinyglass_training_v4_configurable_las"


def _tinyglass_score_distribution(scores: torch.Tensor) -> dict[str, int | float | None]:
    scores = scores.detach().double().cpu().flatten()
    if scores.numel() == 0:
        return {
            "count": 0, "min": None, "max": None,
            "mean": None, "std": None,
            "greater_than_0_99_fraction": None,
            "greater_than_0_999_fraction": None,
        }
    return {
        "count": scores.numel(),
        "min": scores.min().item(),
        "max": scores.max().item(),
        "mean": scores.mean().item(),
        "std": scores.std(unbiased=False).item(),
        "greater_than_0_99_fraction": (scores > 0.99).double().mean().item(),
        "greater_than_0_999_fraction": (scores > 0.999).double().mean().item(),
    }


@torch.no_grad()
def _tinyglass_validation_diagnostics(
    model, loader, device: torch.device, max_batches: int | None,
) -> dict[str, Any]:
    if loader is None:
        raise ValueError(
            "TinyGLASS best-checkpoint selection requires validation_loader"
        )
    model.eval()
    score_chunks: list[torch.Tensor] = []
    label_chunks: list[torch.Tensor] = []
    for index, batch in enumerate(loader):
        if max_batches is not None and index >= max_batches:
            break
        prediction = model.predict(batch["image"].to(device, non_blocking=True))
        score_chunks.append(prediction.anomaly_score.detach().cpu())
        label_chunks.append(torch.as_tensor(batch["label"]).int().cpu())
    return _tinyglass_score_diagnostics(score_chunks, label_chunks)


def _tinyglass_score_diagnostics(
    score_chunks: list[torch.Tensor], label_chunks: list[torch.Tensor],
) -> dict[str, Any]:
    if not score_chunks:
        raise ValueError("validation_loader produced no images")
    scores = torch.cat(score_chunks).flatten()
    labels = torch.cat(label_chunks).flatten()
    if scores.shape != labels.shape:
        raise ValueError("Validation image scores and labels have different sizes")
    if not torch.isfinite(scores).all():
        raise ValueError("TinyGLASS validation scores must be finite")
    if not torch.all((labels == 0) | (labels == 1)):
        raise ValueError("TinyGLASS validation labels must be binary")
    if not torch.any(labels == 0) or not torch.any(labels == 1):
        raise ValueError("TinyGLASS validation requires clean and anomaly samples")
    return {
        "image_auroc": _image_auc(scores.tolist(), labels.tolist()),
        "raw_image_tinyglass_score_distribution": {
            "all": _tinyglass_score_distribution(scores),
            "clean": _tinyglass_score_distribution(scores[labels == 0]),
            "anomaly": _tinyglass_score_distribution(scores[labels == 1]),
        },
    }


@torch.no_grad()
def _tinyglass_build_validation_feature_cache(
    model, loader, device: torch.device, max_batches: int | None,
) -> list[tuple[torch.Tensor, torch.Tensor]]:
    if loader is None:
        raise ValueError(
            "TinyGLASS best-checkpoint selection requires validation_loader"
        )
    model.eval()
    cache: list[tuple[torch.Tensor, torch.Tensor]] = []
    for index, batch in enumerate(loader):
        if max_batches is not None and index >= max_batches:
            break
        features = model.features(
            batch["image"].to(device, non_blocking=True)
        ).detach().cpu()
        labels = torch.as_tensor(batch["label"]).int().cpu().flatten()
        if features.shape[0] != labels.numel():
            raise ValueError("Validation features and labels have different sizes")
        cache.append((features, labels))
    if not cache:
        raise ValueError("validation_loader produced no images")
    return cache


@torch.no_grad()
def _cached_tinyglass_validation_diagnostics(
    model, cache: list[tuple[torch.Tensor, torch.Tensor]], device: torch.device,
) -> dict[str, Any]:
    model.eval()
    score_chunks: list[torch.Tensor] = []
    label_chunks: list[torch.Tensor] = []
    for features, labels in cache:
        patch_scores = model.discriminator(
            features.to(device, non_blocking=True)
        )
        score_chunks.append(patch_scores.flatten(1).amax(1).cpu())
        label_chunks.append(labels)
    return _tinyglass_score_diagnostics(score_chunks, label_chunks)


class TinyGLASSFeatureExtractor(nn.Module):
    """TinyGLASS ResNet-18 layer2/layer3 patch-grid embedding."""

    def __init__(
        self, *, pretrained: bool, weights_name: str,
        patch_size: int = 3, output_channels_per_layer: int = 64,
        freeze_backbone: bool = True,
        feature_grid_resolution: str = "layer3",
    ) -> None:
        super().__init__()
        backbone = build_explicit_torchvision_backbone(
            "resnet18", pretrained=pretrained, weights_name=weights_name
        )
        self.extractor = create_feature_extractor(
            backbone, return_nodes={"layer2": "layer2", "layer3": "layer3"}
        )
        self.freeze_backbone = bool(freeze_backbone)
        self.extractor.requires_grad_(not self.freeze_backbone)
        self.patch_size = patch_size
        self.output_channels_per_layer = output_channels_per_layer
        if feature_grid_resolution not in {"layer2", "layer3"}:
            raise ValueError(
                "feature_grid_resolution must be either 'layer2' or 'layer3'"
            )
        self.feature_grid_resolution = feature_grid_resolution

    def train(self, mode: bool = True):
        super().train(False if self.freeze_backbone else mode)
        return self

    def _patch_grid(self, feature: torch.Tensor) -> torch.Tensor:
        patches = F.unfold(
            feature, kernel_size=self.patch_size, stride=1,
            padding=self.patch_size // 2,
        )
        return patches.reshape(
            feature.shape[0], -1, feature.shape[-2], feature.shape[-1]
        )

    def _reduce(self, feature: torch.Tensor) -> torch.Tensor:
        channels = feature.shape[1]
        if channels % self.output_channels_per_layer:
            raise ValueError(
                f"Patch channels {channels} are not divisible by "
                f"{self.output_channels_per_layer}"
            )
        return feature.reshape(
            feature.shape[0], self.output_channels_per_layer,
            channels // self.output_channels_per_layer,
            *feature.shape[-2:],
        ).mean(2)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        if self.freeze_backbone:
            self.extractor.eval()
            with torch.no_grad():
                features = self.extractor(images)
        else:
            features = self.extractor(images)
        layer2 = self._reduce(self._patch_grid(features["layer2"]))
        layer3 = self._reduce(self._patch_grid(features["layer3"]))
        if self.feature_grid_resolution == "layer2":
            layer3 = F.interpolate(
                layer3, layer2.shape[-2:], mode="bilinear", align_corners=False
            )
        else:
            layer2 = F.adaptive_avg_pool2d(layer2, layer3.shape[-2:])
        return torch.cat((layer2, layer3), dim=1)


class TinyGLASSDiscriminator(nn.Module):
    def __init__(self, input_channels: int = 128, hidden_channels: int = 512) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(input_channels, hidden_channels, 1),
            nn.BatchNorm2d(hidden_channels),
            nn.LeakyReLU(0.2),
            nn.Conv2d(hidden_channels, 1, 1, bias=False),
            nn.Sigmoid(),
        )
        self.apply(self._initialize)

    @staticmethod
    def _initialize(module: nn.Module) -> None:
        if isinstance(module, nn.Conv2d):
            nn.init.normal_(module.weight, 0.0, 0.02)
        elif isinstance(module, nn.BatchNorm2d):
            nn.init.normal_(module.weight, 1.0, 0.02)
            nn.init.zeros_(module.bias)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features).squeeze(1)


class DTDTextureSampler:
    """Texture loader and random augmentation used by TinyGLASS LAS."""

    _EXTENSIONS = {".jpg", ".jpeg", ".png"}

    def __init__(self, root: str | None, input_size: tuple[int, int]) -> None:
        self.root = None if root is None else Path(root)
        self.input_size = input_size
        self.paths = [] if self.root is None else sorted(
            path for path in self.root.rglob("*")
            if path.suffix.lower() in self._EXTENSIONS
        )

    @staticmethod
    def _random_operations(image: Image.Image) -> Image.Image:
        operations = [
            lambda value: TF.adjust_contrast(value, random.uniform(0.8, 1.2)),
            lambda value: TF.adjust_brightness(value, random.uniform(0.8, 1.2)),
            lambda value: TF.adjust_hue(
                TF.adjust_saturation(value, random.uniform(0.8, 1.2)),
                random.uniform(-0.2, 0.2),
            ),
            TF.hflip,
            TF.vflip,
            lambda value: TF.rgb_to_grayscale(value, num_output_channels=3),
            TF.autocontrast,
            TF.equalize,
            lambda value: TF.rotate(
                value, random.uniform(-45, 45),
                interpolation=InterpolationMode.BILINEAR,
            ),
        ]
        for index in random.sample(range(len(operations)), 3):
            image = operations[index](image)
        return image

    def sample(self, batch_size: int, *, device: torch.device) -> torch.Tensor:
        if not self.paths:
            raise RuntimeError("TinyGLASS requires DTD textures for LAS")
        values = []
        for index in torch.randint(len(self.paths), (batch_size,)).tolist():
            with Image.open(self.paths[index]) as image:
                image = TF.resize(
                    image.convert("RGB"), list(self.input_size),
                    interpolation=InterpolationMode.BILINEAR, antialias=True,
                )
                image = self._random_operations(image)
                tensor = pil_to_tensor(image).float().div_(255.0)
            mean = tensor.new_tensor((0.485, 0.456, 0.406)).view(3, 1, 1)
            std = tensor.new_tensor((0.229, 0.224, 0.225)).view(3, 1, 1)
            values.append((tensor - mean) / std)
        return torch.stack(values).to(device, non_blocking=True)


class TinyGLASSLAS(nn.Module):
    _MODES = {"texture", "hole", "mixed"}
    # Full-axis fractions relative to the equivalent wheel diameter. They are
    # derived from the Blender hole contract and its 0.485 m reference wheel.
    _AXIS_FRACTIONS = (
        ((0.027 / 0.485, 0.041 / 0.485), (0.010 / 0.485, 0.017 / 0.485)),
        ((0.044 / 0.485, 0.068 / 0.485), (0.015 / 0.485, 0.027 / 0.485)),
        ((0.073 / 0.485, 0.116 / 0.485), (0.022 / 0.485, 0.041 / 0.485)),
    )

    def __init__(
        self, *, input_size: tuple[int, int], blend_mean: float = 0.5,
        blend_std: float = 0.1, mode: str = "texture",
        hole_probability: float = 0.5,
        hole_luminance_range: tuple[float, float] = (0.18, 0.32),
        hole_severity_weights: tuple[float, float, float] = (0.5, 0.4, 0.1),
    ) -> None:
        super().__init__()
        if mode not in self._MODES:
            raise ValueError(f"LAS mode must be one of {sorted(self._MODES)}")
        if not 0 <= hole_probability <= 1:
            raise ValueError("LAS hole_probability must be in [0, 1]")
        if (
            len(hole_luminance_range) != 2
            or not 0 < hole_luminance_range[0] <= hole_luminance_range[1] < 1
        ):
            raise ValueError("LAS hole_luminance_range must satisfy 0 < min <= max < 1")
        if (
            len(hole_severity_weights) != 3
            or any(not math.isfinite(value) or value < 0 for value in hole_severity_weights)
            or sum(hole_severity_weights) <= 0
        ):
            raise ValueError(
                "LAS hole_severity_weights must contain three finite, "
                "non-negative values with a positive sum"
            )
        self.input_size = input_size
        self.blend_mean = blend_mean
        self.blend_std = blend_std
        self.mode = mode
        self.hole_probability = float(hole_probability)
        self.hole_luminance_range = tuple(float(v) for v in hole_luminance_range)
        severity_total = float(sum(hole_severity_weights))
        self.hole_severity_weights = tuple(
            float(value) / severity_total for value in hole_severity_weights
        )

    @property
    def requires_textures(self) -> bool:
        return self.mode in {"texture", "mixed"}

    @staticmethod
    def downsample_mask(
        image_mask: torch.Tensor, output_size: tuple[int, int],
    ) -> torch.Tensor:
        return F.adaptive_max_pool2d(image_mask, output_size=output_size)

    def _one_mask(
        self, device: torch.device, valid_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        height, width = self.input_size
        if valid_mask is not None:
            if valid_mask.shape != (1, height, width):
                raise ValueError(
                    "TinyGLASS LAS target mask must have shape "
                    f"(1, {height}, {width})"
                )
            valid_mask = valid_mask.to(device=device).bool().squeeze(0)
            if not valid_mask.any():
                raise ValueError("TinyGLASS LAS target mask cannot be empty")
        for _ in range(32):
            masks = []
            for _ in range(2):
                scale_y = 2 ** int(torch.randint(0, 6, (1,)).item())
                scale_x = 2 ** int(torch.randint(0, 6, (1,)).item())
                noise = _rand_perlin_2d(
                    (height, width), (scale_y, scale_x), device=device
                )
                angle = float(torch.empty(()).uniform_(-90, 90))
                noise = TF.rotate(
                    noise[None], angle,
                    interpolation=InterpolationMode.BILINEAR,
                )[0]
                masks.append((noise > 0.5).float())
            choice = float(torch.rand(()))
            if choice > 2 / 3:
                mask = torch.clamp(masks[0] + masks[1], 0, 1)
            elif choice > 1 / 3:
                mask = masks[0] * masks[1]
            else:
                mask = masks[0]
            if valid_mask is not None:
                mask = mask * valid_mask
            if mask.any():
                return mask.unsqueeze(0)
        raise RuntimeError(
            "Could not generate a non-empty TinyGLASS Perlin mask inside the "
            "target wheel"
        )

    @staticmethod
    def _uniform(low: float, high: float) -> float:
        return float(torch.empty(()).uniform_(low, high))

    def _hole_contour(
        self, yy: torch.Tensor, xx: torch.Tensor, *, center_y: int,
        center_x: int, semi_long: float, semi_short: float,
    ) -> torch.Tensor:
        angle = self._uniform(-math.pi, math.pi)
        cos_angle, sin_angle = math.cos(angle), math.sin(angle)
        dx, dy = xx - center_x, yy - center_y
        major = (dx * cos_angle + dy * sin_angle) / semi_long
        minor = (-dx * sin_angle + dy * cos_angle) / semi_short
        radius = torch.sqrt(major.square() + minor.square())
        theta = torch.atan2(minor, major)
        profile = int(torch.multinomial(
            torch.tensor((0.45, 0.40, 0.15), device=yy.device), 1
        ).item())
        if profile == 0:  # jagged slit
            frequencies, amplitudes = (3, 5, 7), (0.11, 0.07, 0.04)
        elif profile == 1:  # branched tear: one pronounced, smooth side lobe
            frequencies, amplitudes = (2, 4, 6), (0.09, 0.06, 0.03)
        else:  # peeled window
            frequencies, amplitudes = (2, 3), (0.07, 0.04)
        boundary = torch.ones_like(radius)
        for frequency, amplitude in zip(frequencies, amplitudes):
            phase = self._uniform(-math.pi, math.pi)
            boundary = boundary + amplitude * torch.sin(frequency * theta + phase)
        if profile == 1:
            branch_angle = self._uniform(-math.pi, math.pi)
            angular_distance = torch.atan2(
                torch.sin(theta - branch_angle), torch.cos(theta - branch_angle)
            )
            boundary = boundary + 0.18 * torch.exp(
                -angular_distance.square() / (2 * 0.28 ** 2)
            )
        return radius <= boundary.clamp_min(0.65)

    def _one_hole_mask(
        self, device: torch.device, valid_mask: torch.Tensor,
    ) -> torch.Tensor:
        height, width = self.input_size
        if valid_mask.shape != (1, height, width):
            raise ValueError(
                "TinyGLASS hole LAS target mask must have shape "
                f"(1, {height}, {width})"
            )
        valid = valid_mask.to(device=device).bool().squeeze(0)
        locations = valid.nonzero(as_tuple=False)
        if locations.numel() == 0:
            raise ValueError("TinyGLASS hole LAS target mask cannot be empty")
        wheel_diameter = 2.0 * math.sqrt(float(valid.sum()) / math.pi)
        yy, xx = torch.meshgrid(
            torch.arange(height, device=device, dtype=torch.float32),
            torch.arange(width, device=device, dtype=torch.float32),
            indexing="ij",
        )
        # Re-sample size and centre together: perspective and partial visibility
        # can make the larger physical severity ranges impossible to contain.
        for _ in range(96):
            severity = int(torch.multinomial(
                torch.tensor(self.hole_severity_weights, device=device), 1
            ).item())
            long_range, short_range = self._AXIS_FRACTIONS[severity]
            long_axis = max(4.0, wheel_diameter * self._uniform(*long_range))
            short_axis = max(3.0, wheel_diameter * self._uniform(*short_range))
            ratio = long_axis / short_axis
            if not 2.0 <= ratio <= 4.5:
                continue
            center = locations[int(torch.randint(len(locations), (1,), device=device))]
            mask = self._hole_contour(
                yy, xx, center_y=int(center[0]), center_x=int(center[1]),
                semi_long=long_axis / 2, semi_short=short_axis / 2,
            )
            if mask.sum() >= 6 and not (mask & ~valid).any():
                return mask.float().unsqueeze(0)
        raise RuntimeError(
            "Could not place a connected hole-like LAS anomaly fully inside "
            "the target wheel"
        )

    def _hole_composite(
        self, image: torch.Tensor, mask: torch.Tensor,
    ) -> torch.Tensor:
        mean = image.new_tensor((0.485, 0.456, 0.406)).view(3, 1, 1)
        std = image.new_tensor((0.229, 0.224, 0.225)).view(3, 1, 1)
        rgb = (image * std + mean).clamp(0, 1)
        mask_4d = mask.unsqueeze(0)
        interior = 1 - F.max_pool2d(1 - mask_4d, 3, stride=1, padding=1)
        interior = interior.squeeze(0)
        rim = (mask - interior).clamp(0, 1)
        noise = torch.rand((1, *self.input_size), device=image.device)
        noise = F.avg_pool2d(noise.unsqueeze(0), 9, stride=1, padding=4).squeeze(0)
        noise = 0.85 + 0.30 * noise
        luminance = self._uniform(*self.hole_luminance_range)
        cavity = (rgb * luminance * noise).clamp(0, 1)
        rim_factor = self._uniform(0.45, 0.70)
        altered = cavity * interior + (rgb * rim_factor) * rim
        altered_normalized = (altered - mean) / std
        return image * (1 - mask) + altered_normalized * mask

    def forward(
        self, images: torch.Tensor, textures: torch.Tensor | None = None,
        target_masks: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if target_masks is not None:
            expected_shape = (images.shape[0], 1, *self.input_size)
            if target_masks.shape != expected_shape:
                raise ValueError(
                    "TinyGLASS LAS target masks must match the image batch: "
                    f"expected {expected_shape}, got {tuple(target_masks.shape)}"
                )
        if self.requires_textures and textures is None:
            raise ValueError(f"TinyGLASS LAS mode '{self.mode}' requires textures")
        augmented_images, masks = [], []
        for index in range(images.shape[0]):
            use_hole = self.mode == "hole" or (
                self.mode == "mixed" and float(torch.rand(())) < self.hole_probability
            )
            valid_mask = None if target_masks is None else target_masks[index]
            if use_hole:
                if valid_mask is None:
                    raise ValueError("TinyGLASS hole LAS requires target masks")
                mask = self._one_hole_mask(images.device, valid_mask)
                augmented = self._hole_composite(images[index], mask)
            else:
                mask = self._one_mask(images.device, valid_mask)
                beta = float(torch.normal(
                    self.blend_mean, self.blend_std, size=(), device=images.device
                ).clamp(0.2, 0.8))
                augmented = images[index] * (1 - mask) + (
                    (1 - beta) * textures[index] + beta * images[index]
                ) * mask
            augmented_images.append(augmented)
            masks.append(mask)
        return torch.stack(augmented_images), torch.stack(masks)


def _tinyglass_focal_loss(
    probabilities: torch.Tensor, targets: torch.Tensor,
    gamma: float = 2.0,
) -> torch.Tensor:
    probabilities = probabilities.clamp(1e-5, 1 - 1e-5)
    targets = targets.float()
    pt = probabilities * targets + (1 - probabilities) * (1 - targets)
    return (-(1 - pt).pow(gamma) * torch.log(pt)).mean()


class TinyGLASSDeployment(nn.Module):
    def __init__(
        self, features: TinyGLASSFeatureExtractor,
        discriminator: TinyGLASSDiscriminator,
    ) -> None:
        super().__init__()
        self.features = copy.deepcopy(features)
        self.discriminator = copy.deepcopy(discriminator)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.discriminator(self.features(images)).unsqueeze(1)


class TinyGLASS(AnomalyDetector):
    """TinyGLASS with dedicated LAS, GAS, patch embedding and discriminator."""

    def __init__(
        self, *, pretrained: bool = True,
        weights_name: str = "IMAGENET1K_V1",
        input_size: tuple[int, int] = (256, 256), patch_size: int = 3,
        epochs: int = 640, learning_rate: float = 1e-4,
        weight_decay: float = 1e-2, noise_std: float = 0.015,
        radius_quantile: float = 0.75, hard_mining_quantile: float = 0.5,
        gas_steps: int = 20, gas_step_size: float = 0.001,
        hypersphere_projection: bool = True,
        feature_grid_resolution: str = "layer3",
        max_samples_per_epoch: int | None = 392,
        texture_root: str | None = None,
        require_texture_dataset: bool = True,
        blend_mean: float = 0.5, blend_std: float = 0.1,
        gaussian_sigma: float = 4.0,
        las_restrict_to_target_mask: bool = False,
        las_mode: str = "texture", las_hole_probability: float = 0.5,
        las_hole_luminance_range: tuple[float, float] = (0.18, 0.32),
        las_hole_severity_weights: tuple[float, float, float] = (0.5, 0.4, 0.1),
        freeze_backbone: bool = True,
        backbone_learning_rate: float = 1e-5,
        fixed_training_duration: bool = False,
        validation_interval: int = 1, validation_batches: int | None = None,
        cache_validation_features: bool = True,
        early_stopping_patience: int | None = 20,
        early_stopping_min_delta: float = 0.0,
        checkpoint_interval: int = 1,
    ) -> None:
        super().__init__()
        if not 0 < radius_quantile < 1:
            raise ValueError("radius_quantile must be in (0, 1)")
        if not 0 <= hard_mining_quantile < 1:
            raise ValueError("hard_mining_quantile must be in [0, 1)")
        self.pretrained = pretrained
        self.weights_name = weights_name
        self.input_size = tuple(input_size)
        self.patch_size = patch_size
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.noise_std = noise_std
        self.radius_quantile = radius_quantile
        self.hard_mining_quantile = hard_mining_quantile
        self.gas_steps = gas_steps
        self.gas_step_size = gas_step_size
        self.hypersphere_projection = hypersphere_projection
        if feature_grid_resolution not in {"layer2", "layer3"}:
            raise ValueError(
                "feature_grid_resolution must be either 'layer2' or 'layer3'"
            )
        self.feature_grid_resolution = feature_grid_resolution
        self.max_samples_per_epoch = max_samples_per_epoch
        self.require_texture_dataset = require_texture_dataset
        self.blend_mean = blend_mean
        self.blend_std = blend_std
        self.gaussian_sigma = gaussian_sigma
        self.las_restrict_to_target_mask = bool(las_restrict_to_target_mask)
        self.las_mode = str(las_mode)
        self.las_hole_probability = float(las_hole_probability)
        self.las_hole_luminance_range = tuple(las_hole_luminance_range)
        self.las_hole_severity_weights = tuple(las_hole_severity_weights)
        if self.las_mode in {"hole", "mixed"} and not self.las_restrict_to_target_mask:
            raise ValueError(
                "TinyGLASS hole/mixed LAS requires las_restrict_to_target_mask=true"
            )
        self.freeze_backbone = bool(freeze_backbone)
        if backbone_learning_rate <= 0:
            raise ValueError("backbone_learning_rate must be positive")
        self.backbone_learning_rate = float(backbone_learning_rate)
        self.fixed_training_duration = fixed_training_duration
        if validation_interval < 1:
            raise ValueError("validation_interval must be at least 1")
        self.validation_interval = validation_interval
        if validation_batches is not None and validation_batches < 1:
            raise ValueError("validation_batches must be positive or None")
        self.validation_batches = validation_batches
        self.cache_validation_features = bool(cache_validation_features)
        if not self.freeze_backbone and self.cache_validation_features:
            warnings.warn(
                "TinyGLASS validation feature cache disabled because "
                "freeze_backbone=false",
                UserWarning,
                stacklevel=2,
            )
            self.cache_validation_features = False
        if early_stopping_patience is not None and early_stopping_patience < 1:
            raise ValueError("early_stopping_patience must be positive or None")
        if early_stopping_min_delta < 0:
            raise ValueError("early_stopping_min_delta cannot be negative")
        self.early_stopping_patience = early_stopping_patience
        self.early_stopping_min_delta = early_stopping_min_delta
        if checkpoint_interval < 1:
            raise ValueError("checkpoint_interval must be at least 1")
        self.checkpoint_interval = checkpoint_interval

        self.features = TinyGLASSFeatureExtractor(
            pretrained=pretrained, weights_name=weights_name,
            patch_size=patch_size, freeze_backbone=self.freeze_backbone,
            feature_grid_resolution=self.feature_grid_resolution,
        )
        self.discriminator = TinyGLASSDiscriminator()
        self.textures = DTDTextureSampler(texture_root, self.input_size)
        self.las = TinyGLASSLAS(
            input_size=self.input_size,
            blend_mean=blend_mean, blend_std=blend_std,
            mode=self.las_mode,
            hole_probability=self.las_hole_probability,
            hole_luminance_range=self.las_hole_luminance_range,
            hole_severity_weights=self.las_hole_severity_weights,
        )
        self.las_hole_severity_weights = self.las.hole_severity_weights
        self.register_buffer("center", torch.zeros(128))
        self.register_buffer("fitted", torch.tensor(False))
        self.fit_summary: dict[str, Any] = {}

    @property
    def is_fitted(self) -> bool:
        return bool(self.fitted)

    def train(self, mode: bool = True):
        super().train(mode)
        if self.freeze_backbone:
            self.features.eval()
        return self

    def checkpoint_config(self) -> dict[str, Any]:
        return {
            "backbone": "resnet18", "pretrained": self.pretrained,
            "weights_name": self.weights_name, "input_size": list(self.input_size),
            "patch_size": self.patch_size, "target_embedding_dimension": 128,
            "epochs": self.epochs, "learning_rate": self.learning_rate,
            "weight_decay": self.weight_decay, "noise_std": self.noise_std,
            "radius_quantile": self.radius_quantile,
            "hard_mining_quantile": self.hard_mining_quantile,
            "gas_steps": self.gas_steps, "gas_step_size": self.gas_step_size,
            "hypersphere_projection": self.hypersphere_projection,
            "feature_grid_resolution": self.feature_grid_resolution,
            "max_samples_per_epoch": self.max_samples_per_epoch,
            "require_texture_dataset": self.require_texture_dataset,
            "blend_mean": self.blend_mean, "blend_std": self.blend_std,
            "gaussian_sigma": self.gaussian_sigma,
            "las_restrict_to_target_mask": self.las_restrict_to_target_mask,
            "las_mode": self.las_mode,
            "las_hole_probability": self.las_hole_probability,
            "las_hole_luminance_range": list(self.las_hole_luminance_range),
            "las_hole_severity_weights": list(self.las_hole_severity_weights),
            "training_algorithm": TINYGLASS_TRAINING_CHECKPOINT_FORMAT,
            "freeze_backbone": self.freeze_backbone,
            "backbone_learning_rate": self.backbone_learning_rate,
            "fixed_training_duration": self.fixed_training_duration,
            "validation_interval": self.validation_interval,
            "validation_batches": self.validation_batches,
            "cache_validation_features": self.cache_validation_features,
            "early_stopping_patience": self.early_stopping_patience,
            "early_stopping_min_delta": self.early_stopping_min_delta,
        }

    def _training_state(self) -> dict[str, Any]:
        return {
            "features": self.features.state_dict(),
            "discriminator": self.discriminator.state_dict(),
            "center": self.center.detach().clone(),
        }

    def _load_training_state(self, state: dict[str, Any]) -> None:
        self.features.load_state_dict(state["features"])
        self.discriminator.load_state_dict(state["discriminator"])
        self.center.copy_(state["center"].to(self.center.device))

    @torch.no_grad()
    def _compute_center(self, train_loader, device: torch.device) -> int:
        total = torch.zeros_like(self.center, device=device)
        patches = 0
        features_were_training = self.features.training
        self.features.eval()
        try:
            for batch in tqdm(
                train_loader, desc="TinyGLASS feature center",
                unit="batch", leave=False,
            ):
                features = self.features(_clean_images(batch, device))
                flattened = features.permute(0, 2, 3, 1).reshape(-1, 128)
                total += flattened.sum(0)
                patches += flattened.shape[0]
        finally:
            self.features.train(features_were_training)
        if patches == 0:
            raise ValueError("train_loader produced no images")
        self.center.copy_(total / patches)
        return patches

    def _project_gas(
        self, gas: torch.Tensor, true: torch.Tensor,
        center: torch.Tensor, radius: torch.Tensor,
    ) -> torch.Tensor:
        gas_flat = gas.permute(0, 2, 3, 1).reshape(-1, 128)
        true_flat = true.permute(0, 2, 3, 1).reshape(-1, 128)
        base = center if self.hypersphere_projection else true_flat
        lower = radius if self.hypersphere_projection else gas_flat.new_tensor(0.5)
        vector = gas_flat - base
        norm = torch.norm(vector, dim=1).clamp_min(1e-10)
        alpha = torch.clamp(norm, lower, 2 * lower)
        projected = base + vector * (alpha / norm).unsqueeze(1)
        return projected.reshape_as(gas.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

    def _training_loss(
        self, true_features: torch.Tensor, fake_features: torch.Tensor,
        feature_mask: torch.Tensor,
    ) -> torch.Tensor | None:
        batch, channels, height, width = true_features.shape
        true_flat = true_features.permute(0, 2, 3, 1).reshape(-1, channels)
        fake_flat = fake_features.permute(0, 2, 3, 1).reshape(-1, channels)
        mask_flat = feature_mask.reshape(-1).bool()
        center = self.center.reshape(1, -1).expand_as(true_flat)
        true_points = torch.cat((fake_flat[~mask_flat], true_flat), dim=0)
        true_centers = torch.cat((center[~mask_flat], center), dim=0)
        radius = torch.quantile(
            torch.norm(true_points - true_centers, dim=1), self.radius_quantile
        ).detach().clamp_min(1e-6)

        gas = (
            true_features + torch.normal(
                0, self.noise_std, size=true_features.shape,
                device=true_features.device,
            )
        ).detach().requires_grad_(True)
        bce_loss = None
        for gas_step in range(self.gas_steps + 1):
            joint_scores = self.discriminator(torch.cat((true_features, gas), dim=0))
            true_scores, gas_scores = joint_scores.split(batch, dim=0)
            bce_loss = (
                F.binary_cross_entropy(true_scores, torch.zeros_like(true_scores))
                + F.binary_cross_entropy(gas_scores, torch.ones_like(gas_scores))
            )
            if gas_step == self.gas_steps:
                break
            gradient = torch.autograd.grad(
                F.binary_cross_entropy(gas_scores, torch.ones_like(gas_scores)),
                gas,
            )[0]
            gradient = gradient / torch.norm(
                gradient, dim=1, keepdim=True
            ).clamp_min(1e-10)
            gas = (gas + self.gas_step_size * gradient).detach()
            if (gas_step + 1) % 5 == 0:
                gas = self._project_gas(gas, true_features, center, radius)
            gas.requires_grad_(True)

        if not mask_flat.any():
            return None
        if self.hypersphere_projection:
            anomalous = fake_flat[mask_flat]
            anomaly_center = center[mask_flat]
            vector = anomalous - anomaly_center
            norm = torch.norm(vector, dim=1).clamp_min(1e-10)
            alpha = torch.clamp(norm, 2 * radius, 4 * radius)
            fake_flat = fake_flat.clone()
            fake_flat[mask_flat] = anomaly_center + vector * (alpha / norm).unsqueeze(1)
            fake_features = fake_flat.reshape(
                batch, height, width, channels
            ).permute(0, 3, 1, 2)

        fake_scores = self.discriminator(fake_features)
        distance = (fake_scores - feature_mask.squeeze(1)).square()
        if self.hard_mining_quantile > 0:
            threshold = torch.quantile(distance.detach(), self.hard_mining_quantile)
            selected = distance >= threshold
            fake_scores = fake_scores[selected]
            target = feature_mask.squeeze(1)[selected]
        else:
            target = feature_mask.squeeze(1)
        return bce_loss + _tinyglass_focal_loss(fake_scores, target)

    def fit(
        self, train_loader, *, device, validation_loader=None,
        work_dir=None, resume=False,
    ) -> TinyGLASS:
        device = torch.device(device)
        if not self.fixed_training_duration and validation_loader is None:
            raise ValueError(
                "TinyGLASS best-checkpoint selection requires validation_loader"
            )
        self.to(device)
        if self.freeze_backbone:
            optimizer = torch.optim.AdamW(
                self.discriminator.parameters(), lr=self.learning_rate * 2,
                weight_decay=self.weight_decay,
            )
        else:
            optimizer = torch.optim.AdamW(
                [
                    {
                        "params": self.discriminator.parameters(),
                        "lr": self.learning_rate * 2,
                    },
                    {
                        "params": self.features.parameters(),
                        "lr": self.backbone_learning_rate,
                    },
                ],
                weight_decay=self.weight_decay,
            )
        best_auc = -math.inf
        best_state = None
        best_epoch = None
        validation_history: list[dict[str, Any]] = []
        no_improvement_validations = 0
        stopped_early = False
        steps = 0
        skipped_empty_masks = 0
        center_patches = 0
        center_patches_total = 0
        center_recomputations = 0
        start_epoch = 0
        completed_epoch = 0
        validation_cache_images = 0
        validation_cache_bytes = 0
        validation_feature_cache = None
        checkpoint_path = (
            None if work_dir is None else Path(work_dir) / "tinyglass_training.ckpt"
        )
        if resume:
            if checkpoint_path is None or not checkpoint_path.is_file():
                raise FileNotFoundError(
                    "resume=true requires work_dir/tinyglass_training.ckpt"
                )
            checkpoint = torch.load(
                checkpoint_path, map_location="cpu", weights_only=False
            )
            if checkpoint.get("format") != TINYGLASS_TRAINING_CHECKPOINT_FORMAT:
                raise ValueError("Unsupported TinyGLASS training checkpoint")
            saved_config = checkpoint.get("model_config")
            if (
                not isinstance(saved_config, dict)
                or saved_config.get("freeze_backbone") != self.freeze_backbone
            ):
                raise ValueError(
                    "TinyGLASS freeze_backbone mismatch; start a new run with "
                    "resume=false"
                )
            if saved_config != self.checkpoint_config():
                raise ValueError("TinyGLASS training checkpoint config mismatch")
            self._load_training_state(checkpoint["model"])
            optimizer.load_state_dict(checkpoint["optimizer"])
            start_epoch = int(checkpoint["epoch"])
            completed_epoch = start_epoch
            if not 0 <= start_epoch <= self.epochs:
                raise ValueError("Invalid epoch in TinyGLASS training checkpoint")
            steps = int(checkpoint.get("steps", 0))
            skipped_empty_masks = int(checkpoint.get("skipped_empty_masks", 0))
            center_patches = int(checkpoint.get("center_patches", 0))
            center_patches_total = int(
                checkpoint.get("center_patches_total", center_patches)
            )
            center_recomputations = int(
                checkpoint.get("center_recomputations", int(center_patches > 0))
            )
            best_auc = float(checkpoint.get("best_auc", -math.inf))
            best_state = checkpoint.get("best_state")
            best_epoch = checkpoint.get("best_epoch")
            validation_history = list(checkpoint.get("validation_history", []))
            no_improvement_validations = int(
                checkpoint.get("no_improvement_validations", 0)
            )
            stopped_early = bool(checkpoint.get("stopped_early", False))
            validation_cache_images = int(
                checkpoint.get("validation_cache_images", 0)
            )
            validation_cache_bytes = int(
                checkpoint.get("validation_cache_bytes", 0)
            )
            _restore_training_rng_state(checkpoint, train_loader)
        else:
            if self.las.requires_textures and not self.textures.paths:
                raise FileNotFoundError(
                    "TinyGLASS training requires texture_root=dtd/images"
                )
            if self.freeze_backbone:
                center_patches = self._compute_center(train_loader, device)
                center_patches_total += center_patches
                center_recomputations += 1
        if (
            self.las.requires_textures
            and not stopped_early
            and start_epoch < self.epochs
            and not self.textures.paths
        ):
            raise FileNotFoundError(
                "TinyGLASS resume requires the original DTD texture dataset"
            )
        if (
            not stopped_early
            and start_epoch < self.epochs
            and not self.fixed_training_duration
            and self.cache_validation_features
        ):
            validation_feature_cache = _tinyglass_build_validation_feature_cache(
                self, validation_loader, device, self.validation_batches
            )
            validation_cache_images = sum(
                labels.numel() for _, labels in validation_feature_cache
            )
            validation_cache_bytes = sum(
                features.numel() * features.element_size()
                for features, _ in validation_feature_cache
            )
        remaining_epochs = (
            () if stopped_early else range(start_epoch + 1, self.epochs + 1)
        )
        epoch_progress = tqdm(
            remaining_epochs,
            initial=start_epoch, total=self.epochs, desc="TinyGLASS training",
            unit="epoch", dynamic_ncols=True,
        )
        for epoch in epoch_progress:
            completed_epoch = epoch
            if not self.freeze_backbone:
                center_patches = self._compute_center(train_loader, device)
                center_patches_total += center_patches
                center_recomputations += 1
            self.train(True)
            samples = 0
            produced = False
            for batch in train_loader:
                produced = True
                images = _clean_images(batch, device)
                textures = (
                    self.textures.sample(images.shape[0], device=device)
                    if self.las.requires_textures else None
                )
                target_masks = None
                if self.las_restrict_to_target_mask:
                    if "target_mask" not in batch:
                        raise KeyError(
                            "las_restrict_to_target_mask=true requires target_mask "
                            "in every training batch"
                        )
                    target_masks = batch["target_mask"].to(
                        device, non_blocking=True
                    )
                augmented, image_mask = self.las(
                    images, textures, target_masks=target_masks
                )
                if self.freeze_backbone:
                    with torch.no_grad():
                        true_features = self.features(images)
                        fake_features = self.features(augmented)
                else:
                    true_features = self.features(images)
                    fake_features = self.features(augmented)
                feature_mask = self.las.downsample_mask(
                    image_mask, output_size=true_features.shape[-2:]
                )
                loss = self._training_loss(
                    true_features.detach() if self.freeze_backbone else true_features,
                    fake_features.detach() if self.freeze_backbone else fake_features,
                    feature_mask,
                )
                if loss is None:
                    skipped_empty_masks += 1
                    continue
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
                epoch_progress.set_postfix(
                    loss=f"{loss.detach().item():.4f}", refresh=False
                )
                steps += 1
                samples += images.shape[0]
                if self.max_samples_per_epoch and samples >= self.max_samples_per_epoch:
                    break
            if not produced:
                raise ValueError("train_loader produced no images")
            if (
                not self.fixed_training_duration
                and (epoch % self.validation_interval == 0 or epoch == self.epochs)
            ):
                self.fitted.fill_(True)
                diagnostics = (
                    _cached_tinyglass_validation_diagnostics(
                        self, validation_feature_cache, device
                    )
                    if validation_feature_cache is not None
                    else _tinyglass_validation_diagnostics(
                        self, validation_loader, device, self.validation_batches
                    )
                )
                diagnostics = {"epoch": epoch, **diagnostics}
                validation_history.append(diagnostics)
                auc = diagnostics["image_auroc"]
                saturation = diagnostics["raw_image_tinyglass_score_distribution"]["all"]
                epoch_progress.set_postfix(
                    val_auc=f"{auc:.4f}",
                    score_gt_0999=(
                        f"{saturation['greater_than_0_999_fraction']:.3f}"
                    ),
                    refresh=False,
                )
                if auc > best_auc + self.early_stopping_min_delta:
                    best_auc = auc
                    best_epoch = epoch
                    best_state = copy.deepcopy(self._training_state())
                    no_improvement_validations = 0
                else:
                    no_improvement_validations += 1
                if (
                    self.early_stopping_patience is not None
                    and no_improvement_validations >= self.early_stopping_patience
                ):
                    stopped_early = True
            if (
                checkpoint_path is not None
                and (
                    epoch % self.checkpoint_interval == 0
                    or epoch == self.epochs
                    or stopped_early
                )
            ):
                _save_checkpoint_atomic({
                    "format": TINYGLASS_TRAINING_CHECKPOINT_FORMAT,
                    "model_config": self.checkpoint_config(),
                    "model": self._training_state(),
                    "optimizer": optimizer.state_dict(),
                    "epoch": epoch,
                    "steps": steps,
                    "skipped_empty_masks": skipped_empty_masks,
                    "center_patches": center_patches,
                    "center_patches_total": center_patches_total,
                    "center_recomputations": center_recomputations,
                    "best_auc": best_auc,
                    "best_state": best_state,
                    "best_epoch": best_epoch,
                    "validation_history": validation_history,
                    "no_improvement_validations": no_improvement_validations,
                    "stopped_early": stopped_early,
                    "validation_cache_images": validation_cache_images,
                    "validation_cache_bytes": validation_cache_bytes,
                    **_capture_training_rng_state(train_loader),
                }, checkpoint_path)
            if stopped_early:
                break
        if best_state is not None:
            self._load_training_state(best_state)
        self.fitted.fill_(True)
        self.eval()
        self.fit_summary = {
            "epochs": self.epochs, "steps": steps,
            "epochs_completed": completed_epoch,
            "center_patches": center_patches,
            "center_patches_total": center_patches_total,
            "center_recomputations": center_recomputations,
            "skipped_empty_masks": skipped_empty_masks,
            "las_restrict_to_target_mask": self.las_restrict_to_target_mask,
            "las_mode": self.las_mode,
            "las_requires_textures": self.las.requires_textures,
            "las_hole_severity_weights": list(self.las_hole_severity_weights),
            "fixed_training_duration": self.fixed_training_duration,
            "freeze_backbone": self.freeze_backbone,
            "backbone_learning_rate": self.backbone_learning_rate,
            "selected_validation_image_auroc": (
                None if best_state is None else best_auc
            ),
            "selected_epoch": best_epoch,
            "stopped_early": stopped_early,
            "no_improvement_validations": no_improvement_validations,
            "validation_feature_cache_enabled": (
                not self.fixed_training_duration and self.cache_validation_features
            ),
            "validation_cache_images": validation_cache_images,
            "validation_cache_bytes": validation_cache_bytes,
            "validation_history": validation_history,
        }
        if checkpoint_path is not None:
            _save_checkpoint_atomic({
                "format": TINYGLASS_TRAINING_CHECKPOINT_FORMAT,
                "model_config": self.checkpoint_config(),
                "model": self._training_state(),
                "optimizer": optimizer.state_dict(),
                "epoch": completed_epoch,
                "steps": steps,
                "skipped_empty_masks": skipped_empty_masks,
                "center_patches": center_patches,
                "center_patches_total": center_patches_total,
                "center_recomputations": center_recomputations,
                "best_auc": best_auc,
                "best_state": best_state,
                "best_epoch": best_epoch,
                "validation_history": validation_history,
                "no_improvement_validations": no_improvement_validations,
                "stopped_early": stopped_early,
                "validation_cache_images": validation_cache_images,
                "validation_cache_bytes": validation_cache_bytes,
                **_capture_training_rng_state(train_loader),
            }, checkpoint_path)
        return self

    @torch.no_grad()
    def _patch_scores(self, images: torch.Tensor) -> torch.Tensor:
        return self.discriminator(self.features(images)).unsqueeze(1)

    @torch.no_grad()
    def predict_with_raw(self, images: torch.Tensor):
        if not self.is_fitted:
            raise RuntimeError("TinyGLASS must be fitted before predict")
        self.eval()
        patch_scores = self._patch_scores(images)
        anomaly_map = F.interpolate(
            patch_scores, images.shape[-2:], mode="bilinear", align_corners=False
        )
        if self.gaussian_sigma > 0:
            kernel = 2 * math.ceil(3 * self.gaussian_sigma) + 1
            anomaly_map = gaussian_blur(
                anomaly_map, [kernel, kernel],
                [self.gaussian_sigma, self.gaussian_sigma],
            )
        anomaly_score = patch_scores.flatten(1).amax(1)
        prediction = AnomalyPrediction(
            anomaly_score.clamp(0, 1), anomaly_map.clamp(0, 1)
        )
        return prediction, anomaly_score, patch_scores

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        return self.predict_with_raw(images)[0]

    def to_deployment_module(self) -> nn.Module:
        if not self.is_fitted:
            raise RuntimeError("TinyGLASS must be fitted before export")
        return TinyGLASSDeployment(self.features, self.discriminator).eval()

def build_model():
    if MODEL_NAME == "patchcore":
        return PatchCore(
            backbone=PATCHCORE_BACKBONE, pretrained=PATCHCORE_PRETRAINED,
            layers=PATCHCORE_LAYERS,
            coreset_sampling_ratio=PATCHCORE_CORESET_SAMPLING_RATIO,
            num_neighbors=PATCHCORE_NUM_NEIGHBORS,
            patch_size=PATCHCORE_PATCH_SIZE, patch_stride=PATCHCORE_PATCH_STRIDE,
            pretrain_embed_dimension=PATCHCORE_PRETRAIN_EMBED_DIMENSION,
            target_embed_dimension=PATCHCORE_TARGET_EMBED_DIMENSION,
            max_patches_per_image=PATCHCORE_MAX_PATCHES_PER_IMAGE,
            max_training_embeddings=PATCHCORE_MAX_TRAINING_EMBEDDINGS,
            max_memory_bank_size=PATCHCORE_MAX_MEMORY_BANK_SIZE,
            projection_dim=PATCHCORE_PROJECTION_DIM,
            sampling_seed=PATCHCORE_SAMPLING_SEED,
            calibration_quantile=PATCHCORE_CALIBRATION_QUANTILE,
            calibration_batches=PATCHCORE_CALIBRATION_BATCHES,
            distance_query_chunk_size=PATCHCORE_DISTANCE_QUERY_CHUNK_SIZE,
            distance_bank_chunk_size=PATCHCORE_DISTANCE_BANK_CHUNK_SIZE,
            gaussian_sigma=PATCHCORE_GAUSSIAN_SIGMA,
        )
    if MODEL_NAME == "efficientad_s":
        return EfficientAD(**{
            key: value for key, value in SELECTED_MODEL_CONFIG.items()
            if key != "batch_size"
        })
    if MODEL_NAME == "supersimplenet":
        return SuperSimpleNet(**{
            key: value for key, value in SELECTED_MODEL_CONFIG.items()
            if key != "batch_size"
        })
    if MODEL_NAME == "tinyglass":
        return TinyGLASS(**{
            key: value for key, value in SELECTED_MODEL_CONFIG.items()
            if key != "batch_size"
        })
    raise ValueError(f"Unknown model name: {MODEL_NAME}")

def build_optimizer(model):
    # I modelli possiedono il proprio fit; PatchCore resta non parametrico.
    return None

def build_scheduler(optimizer):
    return None


In [ ]:
# Strict, offline restore of a final model.ckpt
EXPECTED_MODEL_CLASSES = {
    "patchcore": "PatchCore",
    "efficientad_s": "EfficientAD",
    "supersimplenet": "SuperSimpleNet",
    "tinyglass": "TinyGLASS",
}


def _load_checkpoint_payload(path: str | Path) -> tuple[Path, dict[str, Any]]:
    checkpoint_path = Path(path).expanduser()
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Final model checkpoint is missing: {checkpoint_path}")
    try:
        payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    except TypeError:  # compatibility with older Kaggle PyTorch releases
        payload = torch.load(checkpoint_path, map_location="cpu")
    if not isinstance(payload, dict):
        raise ValueError("Checkpoint payload must be a dictionary")
    required = {"schema_version", "model_class", "model_config", "model_state_dict"}
    missing = required - set(payload)
    if missing:
        raise ValueError(
            "Expected a final model.ckpt, not a training checkpoint; "
            f"missing keys: {sorted(missing)}"
        )
    if payload["schema_version"] != 2:
        raise ValueError(
            f"Unsupported checkpoint schema {payload['schema_version']!r}; expected 2"
        )
    saved_class = str(payload["model_class"]).rsplit(".", 1)[-1]
    expected_class = EXPECTED_MODEL_CLASSES[MODEL_NAME]
    if saved_class != expected_class:
        raise ValueError(
            f"MODEL_NAME={MODEL_NAME!r} expects {expected_class}, "
            f"but the checkpoint contains {payload['model_class']!r}"
        )
    if not isinstance(payload["model_config"], dict):
        raise ValueError("Checkpoint model_config must be a dictionary")
    return checkpoint_path, payload


def _normalized_saved_model_config(
    saved_config: Mapping[str, Any],
) -> tuple[dict[str, Any], tuple[str, ...]]:
    """Normalize only explicitly audited legacy omissions."""
    normalized = dict(saved_config)
    defaulted = []
    if MODEL_NAME == "efficientad_s":
        # The verified 2026-09-01 EfficientAD final checkpoint predates these
        # two constructor fields in checkpoint_config. Its manifest fixes input
        # 256 and the training notebook used CUDA mixed precision.
        audited_defaults = {
            "input_size": tuple(MODEL_INPUT_SIZE),
            "mixed_precision": bool(SELECTED_MODEL_CONFIG["mixed_precision"]),
        }
        for key, value in audited_defaults.items():
            if key not in normalized:
                normalized[key] = value
                defaulted.append(key)
    return normalized, tuple(defaulted)


def _build_selected_model_offline(saved_config: Mapping[str, Any]) -> AnomalyDetector:
    """Build the selected architecture without downloading initialization weights."""
    if MODEL_NAME == "patchcore":
        detector = PatchCore(
            backbone=PATCHCORE_BACKBONE,
            pretrained=False,
            layers=PATCHCORE_LAYERS,
            coreset_sampling_ratio=PATCHCORE_CORESET_SAMPLING_RATIO,
            num_neighbors=PATCHCORE_NUM_NEIGHBORS,
            patch_size=PATCHCORE_PATCH_SIZE,
            patch_stride=PATCHCORE_PATCH_STRIDE,
            pretrain_embed_dimension=PATCHCORE_PRETRAIN_EMBED_DIMENSION,
            target_embed_dimension=PATCHCORE_TARGET_EMBED_DIMENSION,
            max_patches_per_image=PATCHCORE_MAX_PATCHES_PER_IMAGE,
            max_training_embeddings=PATCHCORE_MAX_TRAINING_EMBEDDINGS,
            max_memory_bank_size=PATCHCORE_MAX_MEMORY_BANK_SIZE,
            projection_dim=PATCHCORE_PROJECTION_DIM,
            sampling_seed=PATCHCORE_SAMPLING_SEED,
            calibration_quantile=PATCHCORE_CALIBRATION_QUANTILE,
            calibration_batches=PATCHCORE_CALIBRATION_BATCHES,
            distance_query_chunk_size=PATCHCORE_DISTANCE_QUERY_CHUNK_SIZE,
            distance_bank_chunk_size=PATCHCORE_DISTANCE_BANK_CHUNK_SIZE,
            gaussian_sigma=PATCHCORE_GAUSSIAN_SIGMA,
        )
    else:
        constructor_config = {
            key: value
            for key, value in SELECTED_MODEL_CONFIG.items()
            if key != "batch_size"
        }
        if MODEL_NAME == "efficientad_s":
            # The final state_dict already contains the complete teacher.
            constructor_config["teacher_weights_path"] = None
            constructor_config["require_teacher_weights"] = False
            detector = EfficientAD(**constructor_config)
            detector.teacher_source = saved_config.get("teacher_source")
        elif MODEL_NAME == "supersimplenet":
            saved_pretrained = bool(constructor_config["pretrained"])
            constructor_config["pretrained"] = False
            detector = SuperSimpleNet(**constructor_config)
            detector.pretrained = saved_pretrained
        elif MODEL_NAME == "tinyglass":
            saved_pretrained = bool(constructor_config["pretrained"])
            constructor_config["pretrained"] = False
            detector = TinyGLASS(**constructor_config)
            detector.pretrained = saved_pretrained
        else:  # guarded in Configuration; retained as a local safety gate
            raise ValueError(f"Unsupported MODEL_NAME: {MODEL_NAME!r}")
    return detector


def load_selected_frozen_model(
    checkpoint_path: str | Path,
    *,
    device: torch.device | str = DEVICE,
) -> tuple[AnomalyDetector, dict[str, Any]]:
    checkpoint_path, payload = _load_checkpoint_payload(checkpoint_path)
    saved_config, legacy_defaulted_fields = _normalized_saved_model_config(
        payload["model_config"]
    )
    metadata = dict(payload.get("metadata") or {})
    preprocessing = dict(metadata.get("preprocessing") or {})
    checkpoint_input_size = tuple(preprocessing.get("input_size") or ())
    if checkpoint_input_size and checkpoint_input_size != tuple(MODEL_INPUT_SIZE):
        raise ValueError(
            "Checkpoint preprocessing input size does not match MODEL_NAME preset: "
            f"{checkpoint_input_size} != {MODEL_INPUT_SIZE}"
        )
    metadata_model = dict(metadata.get("model") or {})
    metadata_model_name = metadata_model.get("name")
    if metadata_model_name and metadata_model_name != MODEL_NAME:
        raise ValueError(
            f"Checkpoint metadata model {metadata_model_name!r} != {MODEL_NAME!r}"
        )
    detector = _build_selected_model_offline(saved_config)
    expected_config = detector.checkpoint_config()
    if saved_config != expected_config:
        raise ValueError(
            "Selected notebook preset does not match the checkpoint exactly. "
            f"saved={saved_config!r}, selected={expected_config!r}"
        )
    detector._prepare_state_dict_for_load(payload["model_state_dict"])
    detector.load_state_dict(payload["model_state_dict"], strict=True)
    if not detector.is_fitted:
        raise ValueError("The final checkpoint contains an unfitted model")
    if hasattr(detector, "fit_summary"):
        detector.fit_summary = dict(metadata.get("fit_summary") or {})
    metadata["real_zero_shot_restore"] = {
        "legacy_defaulted_fields": list(legacy_defaulted_fields),
        "checkpoint_path": str(checkpoint_path),
    }
    detector.to(device).eval()
    print(
        f"Loaded frozen {type(detector).__name__} from {checkpoint_path} "
        f"on {device}; no fit or calibration was run."
    )
    return detector, metadata


In [ ]:
# Authenticated Google Drive download with atomic write and size verification.
DRIVE_SCOPE = "https://www.googleapis.com/auth/drive.file"
DRIVE_FOLDER_MIME = "application/vnd.google-apps.folder"


def get_kaggle_secret(name: str, *, required: bool = False) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(
            f"Missing required Kaggle secret {name!r}. Add the same Google Drive "
            "credentials used by the roadmap notebook."
        )
    return value


def build_drive_service():
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build

    credentials = Credentials(
        token=None,
        refresh_token=get_kaggle_secret("GDRIVE_REFRESH_TOKEN", required=True),
        token_uri="https://oauth2.googleapis.com/token",
        client_id=get_kaggle_secret("GDRIVE_CLIENT_ID", required=True),
        client_secret=get_kaggle_secret("GDRIVE_CLIENT_SECRET", required=True),
        scopes=[DRIVE_SCOPE],
    )
    return build("drive", "v3", credentials=credentials, cache_discovery=False)


def parse_drive_item_id(value: str) -> str:
    from urllib.parse import parse_qs, urlparse

    value = str(value or "").strip()
    if not value:
        raise ValueError("Google Drive ID/URL cannot be empty")
    if "/" not in value and "?" not in value:
        item_id = value
    else:
        parsed = urlparse(value)
        parts = [part for part in parsed.path.split("/") if part]
        item_id = parse_qs(parsed.query).get("id", [None])[0]
        if item_id is None:
            for marker in ("d", "folders"):
                if marker in parts and parts.index(marker) + 1 < len(parts):
                    item_id = parts[parts.index(marker) + 1]
                    break
    if not item_id or not all(
        character.isalnum() or character in "-_" for character in item_id
    ):
        raise ValueError(f"Invalid Google Drive ID/URL: {value!r}")
    return item_id


def drive_children(service, parent_id: str, name: str | None = None) -> list[dict]:
    backslash = chr(92)
    escaped_parent = parent_id.replace("'", backslash + "'")
    query = [f"'{escaped_parent}' in parents", "trashed = false"]
    if name is not None:
        escaped_name = name.replace(backslash, backslash * 2).replace(
            "'", backslash + "'"
        )
        query.append(f"name = '{escaped_name}'")
    return service.files().list(
        q=" and ".join(query),
        spaces="drive",
        fields="files(id,name,mimeType,size,parents)",
        pageSize=1000,
    ).execute(num_retries=3).get("files", [])


def selected_drive_run() -> dict[str, Any]:
    if DRIVE_RUN_FOLDER_ID_OVERRIDE:
        return {
            "model": MODEL_NAME,
            "folder_id": parse_drive_item_id(DRIVE_RUN_FOLDER_ID_OVERRIDE),
            "checkpoint_file_id": None,
            "compatible_with_current_preset": None,
            "comment": "Manual override; strict checkpoint configuration still applies.",
        }
    if SELECTED_DRIVE_RUN is None:
        raise RuntimeError(
            f"No verified Drive checkpoint is registered for {MODEL_NAME}. "
            "Use CHECKPOINT_SOURCE='local' and set MODEL_CHECKPOINT_PATH."
        )
    return dict(SELECTED_DRIVE_RUN)


def resolve_drive_checkpoint_file(service, run: Mapping[str, Any]) -> dict[str, Any]:
    folder_id = parse_drive_item_id(run["folder_id"])
    matches = [
        item
        for item in drive_children(service, folder_id, "model.ckpt")
        if item["mimeType"] != DRIVE_FOLDER_MIME
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one model.ckpt in Drive folder {folder_id}, "
            f"found {len(matches)}"
        )
    item = matches[0]
    expected_file_id = run.get("checkpoint_file_id")
    if expected_file_id and item["id"] != expected_file_id:
        raise ValueError(
            "Drive folder checkpoint ID changed: "
            f"expected {expected_file_id}, found {item['id']}"
        )
    return item


def download_drive_checkpoint(
    service,
    run: Mapping[str, Any],
) -> tuple[Path, dict[str, Any]]:
    from googleapiclient.http import MediaIoBaseDownload

    item = resolve_drive_checkpoint_file(service, run)
    file_id = item["id"]
    destination = DRIVE_CHECKPOINT_CACHE_DIR / MODEL_NAME / f"{file_id}.ckpt"
    destination.parent.mkdir(parents=True, exist_ok=True)
    expected_size = int(item.get("size") or 0)
    if (
        not FORCE_CHECKPOINT_DOWNLOAD
        and destination.is_file()
        and expected_size
        and destination.stat().st_size == expected_size
    ):
        print(f"Reusing size-verified cached checkpoint: {destination}")
        return destination, {
            **item,
            "cached": True,
            "run_folder_id": parse_drive_item_id(run["folder_id"]),
        }

    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)
    request = service.files().get_media(fileId=file_id)
    try:
        with temporary.open("wb") as handle, tqdm(
            total=expected_size or None,
            desc=f"Downloading {MODEL_NAME} model.ckpt",
            unit="B",
            unit_scale=True,
            leave=False,
            dynamic_ncols=True,
        ) as progress:
            downloader = MediaIoBaseDownload(
                handle,
                request,
                chunksize=8 * 1024 * 1024,
            )
            downloaded = 0
            done = False
            while not done:
                status, done = downloader.next_chunk(num_retries=5)
                if status is not None:
                    current = int(status.resumable_progress)
                    progress.update(max(0, current - downloaded))
                    downloaded = current
        if expected_size and temporary.stat().st_size != expected_size:
            raise IOError(
                f"Incomplete Drive checkpoint: {temporary.stat().st_size} / "
                f"{expected_size} bytes"
            )
        os.replace(temporary, destination)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise
    return destination, {
        **item,
        "cached": False,
        "run_folder_id": parse_drive_item_id(run["folder_id"]),
    }


def download_selected_checkpoint_from_drive() -> tuple[Path, dict[str, Any]]:
    run = selected_drive_run()
    if run.get("model") != MODEL_NAME:
        raise ValueError(
            f"Drive run model {run.get('model')!r} does not match {MODEL_NAME!r}"
        )
    service = build_drive_service()
    checkpoint_path, metadata = download_drive_checkpoint(service, run)
    print(
        f"Drive run folder: {parse_drive_item_id(run['folder_id'])} | "
        f"checkpoint file: {metadata['id']} | bytes={metadata.get('size', 'unknown')}"
    )
    return checkpoint_path, {"run": run, "file": metadata}


In [ ]:
# Minimal loader for the fixed Real Mars Rover Wheel AD Kaggle dataset.

def _read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

def load_real_wheel_records(root: str | Path) -> list[dict[str, Any]]:
    """Join the three fixed manifests of Real Mars Rover Wheel AD."""
    root = Path(root)
    samples = _read_csv(root / "real_samples.csv")
    crops = {
        row["image_id"]: row for row in _read_csv(root / "real_crops.csv")
    }
    masks = {
        row["image_id"]: row for row in _read_csv(root / "hole_masks.csv")
    }

    records: list[dict[str, Any]] = []
    for sample_row in samples:
        image_id = sample_row["image_id"]
        crop_row = crops[image_id]
        records.append(
            {
                **sample_row,
                "mask_path": masks.get(image_id, {}).get("mask_path", ""),
                "crop_top": int(crop_row["crop_top"]),
                "crop_left": int(crop_row["crop_left"]),
                "crop_height": int(crop_row["crop_height"]),
                "crop_width": int(crop_row["crop_width"]),
                "crop_policy": crop_row["crop_policy"],
                "crop_version": crop_row["crop_version"],
            }
        )
    return records

class RealWheelCropDataset(Dataset[dict[str, Any]]):
    """Load the held-out MAHLI pilot with aligned per-image wheel crops."""

    def __init__(
        self,
        root: str | Path,
        *,
        output_size: tuple[int, int] = (384, 384),
        normalize_mean: RGBTriplet | None = IMAGENET_MEAN,
        normalize_std: RGBTriplet | None = IMAGENET_STD,
    ) -> None:
        self.root = Path(root)
        self.output_size = output_size
        self.normalize_mean = normalize_mean
        self.normalize_std = normalize_std
        self.records = load_real_wheel_records(self.root)

    def _open_image(self, relative_path: str, mode: str) -> Image.Image:
        with Image.open(self.root / relative_path) as opened:
            return opened.convert(mode)

    @staticmethod
    def _crop_box(record: dict[str, Any]) -> tuple[int, int, int, int]:
        return (
            int(record["crop_top"]),
            int(record["crop_left"]),
            int(record["crop_height"]),
            int(record["crop_width"]),
        )

    def __len__(self) -> int:
        return len(self.records)

    def load_native_crop(
        self, index: int
    ) -> tuple[Image.Image, Image.Image, dict[str, Any]]:
        """Return unresized RGB/mask crops for visual QA."""
        record = self.records[index]
        image = self._open_image(record["local_path"], "RGB")
        if record["mask_path"]:
            mask = self._open_image(record["mask_path"], "L")
        else:
            mask = Image.new("L", image.size, color=0)
        top, left, height, width = self._crop_box(record)
        box = (left, top, left + width, top + height)
        return image.crop(box), mask.crop(box), dict(record)

    def load_original(self, index: int) -> tuple[Image.Image, dict[str, Any]]:
        """Return the original RGB frame and metadata for crop-box audits."""
        record = self.records[index]
        return self._open_image(record["local_path"], "RGB"), dict(record)

    def __getitem__(self, index: int) -> dict[str, Any]:
        image, anomaly_mask, record = self.load_native_crop(index)
        image_tensor = pil_to_tensor(image)
        mask_tensor = pil_to_tensor(anomaly_mask)

        image_tensor = TF.resize(
            image_tensor,
            self.output_size,
            interpolation=InterpolationMode.BILINEAR,
            antialias=True,
        )
        mask_tensor = TF.resize(
            mask_tensor,
            self.output_size,
            interpolation=InterpolationMode.NEAREST,
        )

        image_tensor = TF.convert_image_dtype(image_tensor, torch.float32)
        if self.normalize_mean is not None:
            image_tensor = TF.normalize(
                image_tensor,
                mean=self.normalize_mean,
                std=self.normalize_std,
            )

        return {
            "image": image_tensor,
            "anomaly_mask": mask_tensor,
            "label": torch.tensor(record["condition"] == "hole", dtype=torch.long),
            "has_anomaly_mask": bool(record["mask_path"]),
            "metadata": {
                key: value
                for key, value in record.items()
                if key not in {"local_path", "mask_path"}
            },
        }

    def image_for_display(self, image: torch.Tensor) -> torch.Tensor:
        result = image.detach().clone()
        if self.normalize_mean is not None:
            mean = result.new_tensor(self.normalize_mean).view(3, 1, 1)
            std = result.new_tensor(self.normalize_std).view(3, 1, 1)
            result = result * std + mean
        return result.clamp(0.0, 1.0)


In [ ]:
# Visual QA helpers
def plot_crop_boxes(dataset: RealWheelCropDataset, columns: int = 4) -> None:
    rows = math.ceil(len(dataset) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(4.2 * columns, 3.6 * rows))
    axes = np.asarray(axes).reshape(-1)
    for index, axis in enumerate(axes):
        axis.axis("off")
        if index >= len(dataset):
            continue
        image, record = dataset.load_original(index)
        axis.imshow(image)
        axis.add_patch(Rectangle(
            (record["crop_left"], record["crop_top"]),
            record["crop_width"], record["crop_height"],
            fill=False, edgecolor="#ffe600", linewidth=2.0,
        ))
        axis.set_title(
            f"Sol {record['sol']} · {record['condition']}\n{record['image_id'][-16:]}",
            fontsize=9,
        )
    fig.suptitle("Frozen wheel crops on original MAHLI frames", fontsize=15)
    fig.tight_layout()
    plt.show()


def plot_model_inputs(dataset: RealWheelCropDataset, columns: int = 5) -> None:
    rows = math.ceil(len(dataset) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(3.4 * columns, 3.7 * rows))
    axes = np.asarray(axes).reshape(-1)
    for index, axis in enumerate(axes):
        axis.axis("off")
        if index >= len(dataset):
            continue
        sample = dataset[index]
        record = dataset.records[index]
        image = dataset.image_for_display(sample["image"]).permute(1, 2, 0).cpu().numpy()
        mask = sample["anomaly_mask"][0].cpu().numpy() > 0
        axis.imshow(image)
        if mask.any():
            overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
            overlay[..., 0] = 1.0
            overlay[..., 3] = mask.astype(np.float32) * 0.45
            axis.imshow(overlay)
        axis.set_title(
            f"Sol {record['sol']} · {record['condition']}\n"
            f"left={record['crop_left']} · mask_px={int(mask.sum())}",
            fontsize=9,
        )
    fig.suptitle(f"Model inputs {dataset.output_size} (red = approximate hole mask)", fontsize=15)
    fig.tight_layout()
    plt.show()


In [ ]:
# Metrics and outputs kept in sync with src/anomaly_detection/evaluation.

class ExactBinaryMetrics:
    """Exact binary AUROC and average precision for small score collections."""

    def __init__(self) -> None:
        self._scores: list[torch.Tensor] = []
        self._targets: list[torch.Tensor] = []

    @torch.no_grad()
    def update(self, scores: torch.Tensor, targets: torch.Tensor) -> None:
        scores = scores.detach().reshape(-1).cpu()
        targets = targets.detach().reshape(-1).cpu().bool()
        if scores.numel() != targets.numel():
            raise ValueError("scores and targets must contain the same number of values")
        if not scores.numel():
            return
        if not scores.is_floating_point() or not torch.isfinite(scores).all():
            raise ValueError("scores must contain finite floating-point values")
        self._scores.append(scores.clone())
        self._targets.append(targets.clone())

    def compute(self) -> dict[str, float]:
        if not self._scores:
            raise ValueError("AUROC and average precision require both target classes")
        scores = torch.cat(self._scores)
        targets = torch.cat(self._targets)
        positives = int(targets.sum())
        negatives = targets.numel() - positives
        if positives == 0 or negatives == 0:
            raise ValueError("AUROC and average precision require both target classes")

        order = torch.argsort(scores, descending=True)
        sorted_scores = scores[order]
        sorted_targets = targets[order]
        _, group_counts = torch.unique_consecutive(sorted_scores, return_counts=True)
        group_ends = group_counts.cumsum(0) - 1
        true_positives = sorted_targets.cumsum(0)[group_ends].double()
        false_positives = (~sorted_targets).cumsum(0)[group_ends].double()
        recall = true_positives / positives
        false_positive_rate = false_positives / negatives
        precision = true_positives / (true_positives + false_positives)

        zero = torch.zeros(1, dtype=torch.float64)
        auroc = torch.trapezoid(
            torch.cat((zero, recall)),
            torch.cat((zero, false_positive_rate)),
        )
        recall_increment = recall - torch.cat((zero, recall[:-1]))
        average_precision = (precision * recall_increment).sum()
        return {
            "auroc": float(auroc),
            "average_precision": float(average_precision),
        }

class BinaryHistogramMetrics:
    """Memory-bounded binary AUROC and average precision accumulator."""

    def __init__(self, num_bins: int = 2048) -> None:
        if num_bins < 2:
            raise ValueError("num_bins must be at least 2")
        self.num_bins = int(num_bins)
        self.positive_histogram = torch.zeros(num_bins, dtype=torch.int64)
        self.negative_histogram = torch.zeros(num_bins, dtype=torch.int64)

    @torch.no_grad()
    def update(
        self,
        scores: torch.Tensor,
        targets: torch.Tensor,
        valid_mask: torch.Tensor | None = None,
    ) -> None:
        scores = scores.detach().reshape(-1).cpu()
        targets = targets.detach().reshape(-1).cpu().bool()
        if scores.numel() != targets.numel():
            raise ValueError("scores and targets must contain the same number of values")
        if valid_mask is not None:
            valid_mask = valid_mask.detach().reshape(-1).cpu().bool()
            if valid_mask.numel() != scores.numel():
                raise ValueError("valid_mask must match scores")
            scores = scores[valid_mask]
            targets = targets[valid_mask]
        if not scores.numel():
            return
        if not scores.is_floating_point() or not torch.isfinite(scores).all():
            raise ValueError("scores must contain finite floating-point values")
        if scores.min() < 0 or scores.max() > 1:
            raise ValueError("scores must be normalized to [0, 1]")

        bins = (scores * self.num_bins).long().clamp(max=self.num_bins - 1)
        self.positive_histogram += torch.bincount(
            bins[targets], minlength=self.num_bins
        )
        self.negative_histogram += torch.bincount(
            bins[~targets], minlength=self.num_bins
        )

    def compute(self) -> dict[str, float]:
        positives = int(self.positive_histogram.sum())
        negatives = int(self.negative_histogram.sum())
        if positives == 0 or negatives == 0:
            raise ValueError("AUROC and average precision require both target classes")

        true_positives = self.positive_histogram.flip(0).cumsum(0).double()
        false_positives = self.negative_histogram.flip(0).cumsum(0).double()
        recall = true_positives / positives
        false_positive_rate = false_positives / negatives
        precision = true_positives / (true_positives + false_positives).clamp_min(1)

        zero = torch.zeros(1, dtype=torch.float64)
        auroc = torch.trapezoid(
            torch.cat((zero, recall)),
            torch.cat((zero, false_positive_rate)),
        )
        recall_increment = recall - torch.cat((zero, recall[:-1]))
        average_precision = (precision * recall_increment).sum()
        return {
            "auroc": float(auroc),
            "average_precision": float(average_precision),
        }

def _connected_components(mask: torch.Tensor) -> list[list[int]]:
    """Return 4-connected positive regions as flattened pixel indices."""
    mask = mask.detach().cpu().bool().squeeze()
    if mask.ndim != 2:
        raise ValueError("PRO masks must be two-dimensional after squeezing")
    height, width = mask.shape
    remaining = set(torch.nonzero(mask.flatten(), as_tuple=False).flatten().tolist())
    components: list[list[int]] = []
    while remaining:
        start = remaining.pop()
        queue = deque([start])
        component = [start]
        while queue:
            index = queue.popleft()
            row, column = divmod(index, width)
            neighbours = []
            if row > 0:
                neighbours.append(index - width)
            if row + 1 < height:
                neighbours.append(index + width)
            if column > 0:
                neighbours.append(index - 1)
            if column + 1 < width:
                neighbours.append(index + 1)
            for neighbour in neighbours:
                if neighbour in remaining:
                    remaining.remove(neighbour)
                    queue.append(neighbour)
                    component.append(neighbour)
        components.append(component)
    return components

class PerRegionOverlap:
    """Memory-bounded PRO curve and normalized AUPRO accumulator."""

    def __init__(self, num_bins: int = 256, max_fpr: float = 0.30) -> None:
        if num_bins < 2:
            raise ValueError("num_bins must be at least 2")
        if not 0 < max_fpr <= 1:
            raise ValueError("max_fpr must be in (0, 1]")
        self.num_bins = int(num_bins)
        self.max_fpr = float(max_fpr)
        self.normal_histogram = torch.zeros(num_bins, dtype=torch.int64)
        self.region_overlap_sum = torch.zeros(num_bins, dtype=torch.float64)
        self.num_regions = 0

    @torch.no_grad()
    def update(
        self,
        scores: torch.Tensor,
        anomaly_masks: torch.Tensor,
        valid_mask: torch.Tensor | None = None,
    ) -> None:
        scores = scores.detach().cpu()
        anomaly_masks = anomaly_masks.detach().cpu() > 0
        if scores.ndim == 3:
            scores = scores.unsqueeze(1)
        if anomaly_masks.ndim == 3:
            anomaly_masks = anomaly_masks.unsqueeze(1)
        if scores.shape != anomaly_masks.shape:
            raise ValueError("scores and anomaly_masks must have matching shapes")
        if valid_mask is None:
            valid_mask = torch.ones_like(anomaly_masks, dtype=torch.bool)
        else:
            valid_mask = valid_mask.detach().cpu() > 0
            if valid_mask.ndim == 3:
                valid_mask = valid_mask.unsqueeze(1)
            if valid_mask.shape != scores.shape:
                raise ValueError("valid_mask must match scores")

        for score, anomaly_mask, valid in zip(scores, anomaly_masks, valid_mask):
            score = score.squeeze(0)
            anomaly_mask = anomaly_mask.squeeze(0) & valid.squeeze(0)
            valid = valid.squeeze(0)
            normal_scores = score[valid & ~anomaly_mask]
            if normal_scores.numel():
                bins = (normal_scores * self.num_bins).long().clamp(
                    min=0, max=self.num_bins - 1
                )
                self.normal_histogram += torch.bincount(
                    bins, minlength=self.num_bins
                )
            flattened_scores = score.flatten()
            for component in _connected_components(anomaly_mask):
                component_scores = flattened_scores[component]
                bins = (component_scores * self.num_bins).long().clamp(
                    min=0, max=self.num_bins - 1
                )
                histogram = torch.bincount(bins, minlength=self.num_bins)
                self.region_overlap_sum += (
                    histogram.flip(0).cumsum(0).double() / len(component)
                )
                self.num_regions += 1

    @staticmethod
    def _normalized_area(
        fpr: torch.Tensor,
        pro: torch.Tensor,
        max_fpr: float,
    ) -> float:
        below = fpr <= max_fpr
        clipped_fpr = fpr[below]
        clipped_pro = pro[below]
        if clipped_fpr[-1] < max_fpr:
            upper_index = int(torch.nonzero(fpr > max_fpr)[0])
            lower_index = upper_index - 1
            denominator = fpr[upper_index] - fpr[lower_index]
            weight = (
                0.0
                if denominator == 0
                else float((max_fpr - fpr[lower_index]) / denominator)
            )
            interpolated = pro[lower_index] + weight * (
                pro[upper_index] - pro[lower_index]
            )
            clipped_fpr = torch.cat(
                (clipped_fpr, torch.tensor([max_fpr], dtype=torch.float64))
            )
            clipped_pro = torch.cat((clipped_pro, interpolated.reshape(1)))
        area = torch.trapezoid(clipped_pro, clipped_fpr) / max_fpr
        return float(area)

    def compute(self) -> dict[str, Any]:
        total_normal = int(self.normal_histogram.sum())
        if total_normal == 0 or self.num_regions == 0:
            raise ValueError("PRO requires normal pixels and at least one anomaly region")
        fpr = self.normal_histogram.flip(0).cumsum(0).double() / total_normal
        pro = self.region_overlap_sum / self.num_regions
        fpr = torch.cat((torch.zeros(1, dtype=torch.float64), fpr))
        pro = torch.cat((torch.zeros(1, dtype=torch.float64), pro))

        reported_max_fprs = sorted({0.05, 0.10, self.max_fpr})
        aupro_by_max_fpr = {
            f"{max_fpr:.2f}": self._normalized_area(fpr, pro, max_fpr)
            for max_fpr in reported_max_fprs
        }
        return {
            "aupro": aupro_by_max_fpr[f"{self.max_fpr:.2f}"],
            "max_fpr": self.max_fpr,
            "aupro_by_max_fpr": aupro_by_max_fpr,
            "num_regions": self.num_regions,
            "curve": [
                {"false_positive_rate": float(x), "pro": float(y)}
                for x, y in zip(fpr, pro)
            ],
        }

def _batch_metadata_value(
    metadata: Mapping[str, Any], key: str, index: int
) -> Any:
    values = metadata.get(key)
    if values is None:
        return ""
    if isinstance(values, torch.Tensor):
        value = values[index]
        return value.item() if value.ndim == 0 else value.tolist()
    value = values[index]
    return value.item() if isinstance(value, torch.Tensor) else value

@torch.no_grad()
def evaluate_real_wheel_pilot(
    detector: AnomalyDetector,
    loader: Iterable[Mapping[str, Any]],
    *,
    device: torch.device | str,
    image_for_display: Callable[[torch.Tensor], torch.Tensor],
    histogram_bins: int = 2048,
    pro_bins: int = 256,
    pro_max_fpr: float = 0.30,
    description: str = "Real-wheel zero-shot inference",
) -> dict[str, Any]:
    """Evaluate a frozen detector once on every real-wheel crop."""
    if not detector.is_fitted:
        raise RuntimeError("The anomaly detector must be fitted before evaluation")

    device = torch.device(device)
    detector.to(device)
    detector.train(False)
    image_metric = ExactBinaryMetrics()
    pixel_metric = BinaryHistogramMetrics(histogram_bins)
    pro_metric = PerRegionOverlap(pro_bins, pro_max_fpr)
    records: list[dict[str, Any]] = []
    samples: list[dict[str, Any]] = []
    scores_at_zero = 0
    scores_at_one = 0

    for batch in tqdm(loader, desc=description):
        images = torch.as_tensor(batch["image"])
        labels = torch.as_tensor(batch["label"]).reshape(-1).cpu()
        masks = torch.as_tensor(batch["anomaly_mask"])
        if images.ndim != 4 or images.shape[0] != labels.numel():
            raise ValueError("image batch and labels have incompatible shapes")
        if images.shape[0] != 1:
            raise ValueError(
                "The real zero-shot protocol requires batch_size=1 so each "
                "frame is inferred independently"
            )

        images_device = images.to(device, non_blocking=True)
        predict_with_raw = getattr(detector, "predict_with_raw", None)
        if predict_with_raw is None:
            prediction = detector.predict(images_device)
            raw_image_scores = prediction.anomaly_score
        else:
            prediction, raw_image_scores, _ = predict_with_raw(images_device)
        raw_image_scores = raw_image_scores.detach().reshape(-1).cpu()
        if raw_image_scores.shape != labels.shape:
            raise ValueError("raw image scores must match labels")
        if prediction.anomaly_map.shape != masks.shape:
            raise ValueError(
                "predicted anomaly maps and approximate masks must have "
                f"matching shapes: {prediction.anomaly_map.shape} != {masks.shape}"
            )

        image_metric.update(raw_image_scores, labels)
        pixel_metric.update(prediction.anomaly_map, masks)
        pro_metric.update(prediction.anomaly_map, masks)
        normalized_scores = prediction.anomaly_score.detach().reshape(-1).cpu()
        scores_at_zero += int((normalized_scores == 0).sum())
        scores_at_one += int((normalized_scores == 1).sum())
        metadata = batch.get("metadata", {})

        for index, label_tensor in enumerate(labels):
            label = int(label_tensor)
            image_id = str(_batch_metadata_value(metadata, "image_id", index))
            condition = str(_batch_metadata_value(metadata, "condition", index))
            sol = _batch_metadata_value(metadata, "sol", index)
            anomaly_map = prediction.anomaly_map[index].detach().cpu()
            record = {
                "image_id": image_id,
                "sol": int(sol) if str(sol) else None,
                "condition": condition or ("hole" if label else "no_visible_hole"),
                "label": label,
                "raw_anomaly_score": float(raw_image_scores[index]),
                "normalized_anomaly_score": float(normalized_scores[index]),
                "anomaly_map_min": float(anomaly_map.min()),
                "anomaly_map_mean": float(anomaly_map.mean()),
                "anomaly_map_max": float(anomaly_map.max()),
            }
            records.append(record)
            samples.append(
                {
                    **record,
                    "image": image_for_display(images[index]).detach().cpu(),
                    "anomaly_mask": masks[index].detach().cpu(),
                    "anomaly_map": anomaly_map,
                }
            )

    if not records:
        raise ValueError("Evaluation loader produced no images")
    image_result = image_metric.compute()
    pixel_result = pixel_metric.compute()
    pro_result = pro_metric.compute()
    positive_pixels = int(pixel_metric.positive_histogram.sum())
    negative_pixels = int(pixel_metric.negative_histogram.sum())
    total_pixels = positive_pixels + negative_pixels
    saturated = scores_at_zero + scores_at_one
    metrics = {
        "image_auroc": image_result["auroc"],
        "image_average_precision": image_result["average_precision"],
        "pixel_auroc": pixel_result["auroc"],
        "pixel_average_precision": pixel_result["average_precision"],
        "aupro": pro_result["aupro"],
        "aupro_max_fpr": pro_result["max_fpr"],
        "aupro_by_max_fpr": pro_result["aupro_by_max_fpr"],
        "pro_num_regions": pro_result["num_regions"],
        "random_pixel_ap_baseline": positive_pixels / total_pixels,
        "approximate_anomaly_pixel_fraction": positive_pixels / total_pixels,
        "num_images": len(records),
        "num_clean_images": sum(record["label"] == 0 for record in records),
        "num_hole_images": sum(record["label"] == 1 for record in records),
        "normalized_image_scores_at_zero": scores_at_zero,
        "normalized_image_scores_at_one": scores_at_one,
        "normalized_image_score_saturation_fraction": saturated / len(records),
        "pixel_metric_domain": "entire_resized_crop",
        "pixel_ground_truth": "approximate_hole_masks",
        "target_wheel_metrics_available": False,
    }
    return {
        "metrics": metrics,
        "records": records,
        "samples": samples,
        "pro_curve": pro_result["curve"],
    }

def plot_real_wheel_predictions(
    samples: list[dict[str, Any]],
    *,
    title: str,
    colormap: str = "magma",
    overlay_alpha: float = 0.55,
) -> Figure:
    """Plot the real crop, normalized anomaly map and overlay for each frame."""
    if not samples:
        raise ValueError("samples cannot be empty")
    if not 0 <= overlay_alpha <= 1:
        raise ValueError("overlay_alpha must be in [0, 1]")

    figure = Figure(figsize=(14, 4.1 * len(samples)), layout="constrained")
    FigureCanvasAgg(figure)
    axes = figure.subplots(len(samples), 3, squeeze=False)
    heatmap_artist = None
    for row, sample in enumerate(samples):
        image = sample["image"].permute(1, 2, 0).numpy()
        anomaly_mask = sample["anomaly_mask"].squeeze().numpy() > 0
        anomaly_map = sample["anomaly_map"].squeeze().numpy()
        image_id = sample.get("image_id") or "unknown"
        condition = sample.get("condition") or (
            "hole" if sample.get("label") else "no_visible_hole"
        )
        score_text = (
            f"raw={sample['raw_anomaly_score']:.4g} | "
            f"normalized={sample['normalized_anomaly_score']:.3f}"
        )

        axes[row, 0].imshow(image)
        axes[row, 0].set_title(f"{image_id} — {condition}\n{score_text}")
        heatmap_artist = axes[row, 1].imshow(
            anomaly_map, cmap=colormap, vmin=0, vmax=1
        )
        axes[row, 1].set_title("Normalized anomaly map")
        axes[row, 2].imshow(image)
        axes[row, 2].imshow(
            anomaly_map,
            cmap=colormap,
            vmin=0,
            vmax=1,
            alpha=overlay_alpha,
        )
        if anomaly_mask.any() and not anomaly_mask.all():
            for axis in axes[row, 1:]:
                axis.contour(
                    anomaly_mask,
                    levels=[0.5],
                    colors="lime",
                    linewidths=1.4,
                )
        axes[row, 2].set_title("Overlay\nlime=approximate hole boundary")
        for axis in axes[row]:
            axis.axis("off")

    figure.suptitle(title, fontsize=15)
    figure.colorbar(
        heatmap_artist,
        ax=axes[:, 1:].ravel().tolist(),
        label="Normalized anomaly score",
        shrink=0.8,
    )
    return figure

def save_real_wheel_pilot_results(
    evaluation: Mapping[str, Any],
    output_dir: str | Path,
    *,
    context: Mapping[str, Any],
    rows_per_figure: int = 4,
    colormap: str = "magma",
    overlay_alpha: float = 0.55,
    dpi: int = 150,
) -> tuple[list[Figure], dict[str, Any]]:
    """Persist metrics, per-image scores, PRO curve and paginated figures."""
    if rows_per_figure < 1:
        raise ValueError("rows_per_figure must be at least 1")
    if dpi < 1:
        raise ValueError("dpi must be at least 1")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    metrics_path = output_dir / "metrics.json"
    metrics_payload = {"context": dict(context), "metrics": evaluation["metrics"]}
    metrics_path.write_text(
        json.dumps(metrics_payload, indent=2) + "\n", encoding="utf-8"
    )

    records = list(evaluation["records"])
    scores_path = output_dir / "image_scores.csv"
    with scores_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

    pro_curve_path = output_dir / "pro_curve.csv"
    with pro_curve_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["false_positive_rate", "pro"])
        writer.writeheader()
        writer.writerows(evaluation["pro_curve"])

    figures: list[Figure] = []
    figure_paths: list[Path] = []
    samples = list(evaluation["samples"])
    for start in range(0, len(samples), rows_per_figure):
        page = start // rows_per_figure + 1
        page_samples = samples[start : start + rows_per_figure]
        figure = plot_real_wheel_predictions(
            page_samples,
            title=f"{context['model_name']} real-wheel zero-shot — page {page}",
            colormap=colormap,
            overlay_alpha=overlay_alpha,
        )
        figure_path = output_dir / f"predictions_page_{page:02d}.png"
        figure.savefig(figure_path, dpi=dpi, bbox_inches="tight")
        figures.append(figure)
        figure_paths.append(figure_path)

    return figures, {
        "metrics": metrics_path,
        "image_scores": scores_path,
        "pro_curve": pro_curve_path,
        "figures": figure_paths,
    }


In [ ]:
# Load the fixed Kaggle dataset and build the one-image-at-a-time loader.
normalization = (IMAGENET_MEAN, IMAGENET_STD) if NORMALIZE_IMAGENET else (None, None)
real_dataset = RealWheelCropDataset(
    DATASET_ROOT,
    output_size=MODEL_INPUT_SIZE,
    normalize_mean=normalization[0],
    normalize_std=normalization[1],
)
real_loader = DataLoader(
    real_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)
print(f"Loaded {len(real_dataset)} samples from {DATASET_ROOT}")


In [ ]:
# Preview the frozen crop boxes on the original frames.
plot_crop_boxes(real_dataset, columns=4)


In [ ]:
# Preview the exact resized inputs and approximate hole masks.
plot_model_inputs(real_dataset, columns=5)


In [ ]:
# Download/restore the selected frozen model. No real image is passed here.
detector = None
checkpoint_metadata = {}
drive_checkpoint_info = None
print(
    f"Selected preset: {MODEL_NAME} | model input: {MODEL_INPUT_SIZE} | "
    f"pilot batch size: {BATCH_SIZE} | checkpoint source: {CHECKPOINT_SOURCE}"
)

if CHECKPOINT_SOURCE == "none":
    print("Checkpoint restore disabled; no real-image inference can run.")
elif CHECKPOINT_SOURCE == "local":
    if MODEL_CHECKPOINT_PATH is None:
        raise ValueError(
            "CHECKPOINT_SOURCE='local' requires MODEL_CHECKPOINT_PATH"
        )
    detector, checkpoint_metadata = load_selected_frozen_model(
        MODEL_CHECKPOINT_PATH,
        device=DEVICE,
    )
else:
    downloaded_checkpoint, drive_checkpoint_info = (
        download_selected_checkpoint_from_drive()
    )
    detector, checkpoint_metadata = load_selected_frozen_model(
        downloaded_checkpoint,
        device=DEVICE,
    )


## Frozen one-image-at-a-time inference

The selected synthetic checkpoint is evaluated once on each of the 20 held-out
real crops. Image AUROC/AP use raw image scores; Pixel AUROC/AP and AUPRO use
the normalized anomaly maps and approximate hole masks. Pixel metrics cover the
entire resized crop because these real frames do not include a full-wheel mask.
No real result is used to select a model, checkpoint, score, smoothing value,
threshold or crop.


In [ ]:
# One frozen forward pass per real image; predictions are retained for all outputs.
if detector is None:
    raise RuntimeError(
        "A fitted checkpoint is required. Set CHECKPOINT_SOURCE to google_drive or local."
    )
real_evaluation = evaluate_real_wheel_pilot(
    detector,
    real_loader,
    device=DEVICE,
    image_for_display=real_dataset.image_for_display,
    histogram_bins=METRIC_HISTOGRAM_BINS,
    pro_bins=PRO_BINS,
    pro_max_fpr=PRO_MAX_FPR,
)
print(f"Evaluated {real_evaluation['metrics']['num_images']} real images exactly once.")


In [ ]:
# Save tabular/visual outputs and show every real prediction in the notebook.
restore_info = checkpoint_metadata.get("real_zero_shot_restore", {})
checkpoint_path = Path(restore_info.get("checkpoint_path", "model.ckpt"))
if drive_checkpoint_info is not None:
    checkpoint_file_id = str(drive_checkpoint_info["file"]["id"])
    drive_folder_id = str(drive_checkpoint_info["run"]["folder_id"])
else:
    checkpoint_file_id = checkpoint_path.stem
    drive_folder_id = None
safe_checkpoint_id = "".join(
    character if character.isalnum() or character in "-_" else "_"
    for character in checkpoint_file_id
)
run_output_dir = OUTPUT_ROOT / MODEL_NAME / safe_checkpoint_id
result_context = {
    "model_name": MODEL_NAME,
    "model_input_size": list(MODEL_INPUT_SIZE),
    "checkpoint_source": CHECKPOINT_SOURCE,
    "checkpoint_file_id": checkpoint_file_id,
    "drive_run_folder_id": drive_folder_id,
    "checkpoint_path": str(checkpoint_path),
    "dataset_root": str(DATASET_ROOT),
    "crop_version": real_dataset.records[0]["crop_version"],
    "selection_on_real_data": False,
}
result_figures, result_artifacts = save_real_wheel_pilot_results(
    real_evaluation,
    run_output_dir,
    context=result_context,
    rows_per_figure=VISUALIZATION_ROWS_PER_PAGE,
    colormap=VISUALIZATION_COLORMAP,
    overlay_alpha=VISUALIZATION_OVERLAY_ALPHA,
    dpi=VISUALIZATION_DPI,
)

metrics = real_evaluation["metrics"]
metric_rows = [
    ("Image AUROC", metrics["image_auroc"]),
    ("Image AP", metrics["image_average_precision"]),
    ("Pixel AUROC — full crop", metrics["pixel_auroc"]),
    ("Pixel AP — full crop", metrics["pixel_average_precision"]),
    ("Random pixel-AP baseline", metrics["random_pixel_ap_baseline"]),
]
metric_rows.extend(
    (f"AUPRO @ FPR {cutoff}", value)
    for cutoff, value in metrics["aupro_by_max_fpr"].items()
)
metric_table = pd.DataFrame(metric_rows, columns=["metric", "value"])
display(metric_table.style.format({"value": "{:.4f}"}))

score_table = pd.DataFrame(real_evaluation["records"])
display(
    score_table.style.format(
        {
            "raw_anomaly_score": "{:.5g}",
            "normalized_anomaly_score": "{:.4f}",
            "anomaly_map_min": "{:.4f}",
            "anomaly_map_mean": "{:.4f}",
            "anomaly_map_max": "{:.4f}",
        }
    )
)

for figure in result_figures:
    display(figure)
    plt.close(figure)

print(f"Outputs: {run_output_dir}")
for name, path_or_paths in result_artifacts.items():
    if isinstance(path_or_paths, list):
        print(f"- {name}: {len(path_or_paths)} files")
    else:
        print(f"- {name}: {path_or_paths}")


## Interpretation contract

- The tables and figures are descriptive evidence from a 20-image correlated
  pilot, not a new validation set.
- Lime contours are the approximate human hole annotations; they are shown for
  interpretation only and never enter model inference.
- Pixel AUROC/AP and AUPRO use the full crop. Target-wheel pixel metrics are
  intentionally omitted until a complete wheel mask is available.
- Raw score magnitudes are model-specific; compare rankings and metrics, not
  absolute raw values across different model families.
- Do not rerank checkpoints or alter crop coordinates, score aggregation,
  smoothing or thresholds after inspecting these outputs.
